# MICA benchmark: GPU session `cryozeta_a` (CryoZeta)

This notebook builds atomic models from the cryo-EM maps of the benchmark with **CryoZeta** (Kihara lab; code at commit `c544c6bb`, weights `v0.0.1` from Hugging Face), one entry
after the other, and scores every model with US-align, the same scoring as for the other methods. CryoZeta gets the map, the sequences and the multiple sequence alignments of the sequences
(searched on the public ColabFold server), **no AlphaFold dataset is needed**. The method was added to the benchmark after Prof. Cheng's request of 31 October 2026; the other methods are not run here.

**Hardware.** The authors ask for a GPU with 32 GB of memory; a Kaggle T4 has 15 GB. The notebook keeps the settings of the authors' demo and lowers only the attention chunk size when a run
runs out of GPU memory (four steps, the size changes the memory and the time, not the algorithm). An entry that still does not fit is a result of the method on this hardware (`failed`, "CUDA out of
memory") and stays in the tables. A larger GPU is needed for those entries.

* The Kaggle session has **two GPUs**. The notebook uses both: two programs run at the same time, one on each GPU, so the work of about 2 x 11 hours fits in the session.
* The list has 20 entries, in the order of the seeded selection (the two resolution classes interleaved). No run of CryoZeta has been timed on a T4 yet: the first estimate is
  15 seconds per residue of the deposited model; as soon as one run has finished, the measured rate replaces it. Before an entry starts, the notebook checks that 1.25 times its
  estimated time fits in what is left of the budget (`HOURS` in the first cell). When the budget is nearly used up, no new run starts; what is not done is written to `pending.csv`
  and is run in the next session.
* A run that crashes is a result and stays in the tables (with the reason); a run that the end of the session cuts is **not** a result: it is run again.
* Every run has a time limit of 6 hours.

**What you do (Kaggle, background mode)**: accelerator **GPU T4 x2** (not P100), Internet **On**, no input dataset and no secret, `HOURS` in the first cell (the hours of GPU time you allow), then
*Save Version -> Save & Run All (Commit)*. It runs on Kaggle's servers; you can close the tab and your Wi-Fi does not matter. The setup (environment, weights, TEASER++) takes 20 to 40 minutes
before the first run. When the version shows *Succeeded*, open it and download the zip `gpu_cryozeta_a_result_....zip` from its Output. Send it to me. **Do not stop or cancel the run**: a cancelled run keeps nothing.


## Settings (the only number you may have to change)

In [ ]:
# HOURS = how many hours of Kaggle GPU time this run may use. Kaggle gives about 30 GPU hours per week for free. If you have less than 12 hours left this week,
# type (hours left minus 1) here. The notebook plans its work to end before HOURS are used and never goes above 11.
HOURS = 11.0
# GUARD_MINUTES: a method that crashes (not an out-of-memory kill) within this many minutes on two entries in a row is paused for the rest of the session, because that is probably a
# problem of the installation and not a result. Leave it as it is unless I tell you to change it (0 switches the guard off).
GUARD_MINUTES = 60


## Step 1. Look at the machine

In [ ]:
import csv, datetime, getpass, glob, gzip, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, tarfile, threading, time, zipfile
from collections import deque

IN_KAGGLE = os.path.isdir("/kaggle/working") and not os.environ.get("MICA_WORK")      # MICA_WORK and MICA_BIG are only used by the tests
IN_COLAB = (not IN_KAGGLE) and os.path.isdir("/content") and not os.environ.get("MICA_WORK")
PLATFORM = "kaggle" if IN_KAGGLE else ("colab" if IN_COLAB else "other")
if os.environ.get("MICA_WORK"):
    WORK = OUT_DIR = os.environ["MICA_WORK"]
elif IN_KAGGLE:
    WORK, OUT_DIR = "/kaggle/working/pilot", "/kaggle/working"
elif IN_COLAB:
    WORK, OUT_DIR = "/content/pilot", "/content"
else:
    WORK = OUT_DIR = os.path.abspath("pilot")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
LOG = os.path.join(WORK, "run.log")
REQUIRE_GPU = True
NET_HOSTS = (('PyPI', 'https://pypi.org/simple/uv/', True), ('conda-forge', 'https://conda.anaconda.org/conda-forge/noarch/current_repodata.json', True), ('GitHub', 'https://github.com/kiharalab/CryoZeta', True), ('Hugging Face (CryoZeta weights)', 'https://huggingface.co/api/models/KiharaLab/CryoZeta', True), ('ColabFold server', 'https://api.colabfold.com/', False), ('EBI FTP', 'https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/header/emd-80558.xml', True), ('RCSB', 'https://files.rcsb.org/header/26DK.cif', True))
STATE = {"ok": True, "problem": "", "t0": time.time(), "started": datetime.datetime.now(datetime.timezone.utc).isoformat(),
         "phenix_env": None, "phenix_version": "", "installer_sha256": "", "usalign": "", "usalign_version": "", "weights": {}, "env": {},
         "entries": {}, "gpu": ""}
SECRETS = []          # kept in memory only, to make sure they never end up in the zip


def log(msg=""):
    print(msg, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")


def fail(msg):
    STATE["ok"] = False
    STATE["problem"] = msg
    log("\n" + "!" * 72 + "\nSTOPPED: " + msg + "\nThe steps below are skipped. The last step still packs a zip with the log: send it to me.\n" + "!" * 72)


def run(cmd, env=None, every=1, cwd=None, log_file=None, timeout=None, quiet=False):
    """Runs a command, prints and logs its output as it comes. Returns (exit code, last lines, timed_out).
    env: values are set, None removes the variable. A command that runs longer than `timeout` seconds is killed with its child processes.
    Progress bars (lines with %|) are printed only now and then; quiet=True keeps the output out of the notebook (it still goes to log_file)."""
    e = dict(os.environ)
    for k, v in (env or {}).items():
        if v is None:
            e.pop(k, None)
        else:
            e[k] = v
    e["PYTHONUNBUFFERED"] = "1"
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e, cwd=cwd, errors="replace",
                             start_new_session=True)
    except OSError as ex:
        log("could not start %s: %s" % (cmd[0] if isinstance(cmd, (list, tuple)) else cmd, ex))
        return 127, [str(ex)], False
    timed = {"out": False}

    def killer():
        timed["out"] = True
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except OSError:
            pass

    timer = threading.Timer(timeout, killer) if timeout else None
    if timer:
        timer.start()
    lf = open(log_file, "a", encoding="utf-8") if log_file else None
    last, n, bars = deque(maxlen=40), 0, 0
    try:
        for line in p.stdout:
            line = line.rstrip("\n")
            if len(line) > 600:
                line = line[:600] + " ..."
            last.append(line)
            n += 1
            if lf:
                lf.write(line + "\n")
                lf.flush()                                  # a program that is silent for an hour must not hide its last lines in a buffer
            if "%|" in line:
                bars += 1
                show = bars % 100 == 1
            else:
                show = every == 1 or n <= 5 or n % every == 0
            if show and not quiet:
                log(line)
        p.wait()
    finally:
        if timer:
            timer.cancel()
        if lf:
            lf.close()
        p.stdout.close()
    return p.returncode, list(last), timed["out"]


def sh(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, errors="replace")
        return (r.stdout + r.stderr).strip()
    except Exception as ex:
        return "ERROR: %s" % ex


def secret(name, prompt, hidden=True):
    """Colab Secrets, Kaggle Secrets, an environment variable, or a prompt, in this order. Returns "" if nothing can be found."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    try:
        return (getpass.getpass(prompt) if hidden else input(prompt)).strip()
    except Exception:
        return ""


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def meminfo_gb():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return round(int(line.split()[1]) / 1048576, 1)
    except OSError:
        pass
    return None


def pick_big_dir():
    """The folder for everything large (Phenix, Python environment, MICA): the disk with the most free space."""
    if os.environ.get("MICA_BIG"):
        return os.environ["MICA_BIG"]
    best = None
    for base in ("/kaggle/temp", "/tmp", "/content", WORK):
        if os.path.isdir(base) and os.access(base, os.W_OK):
            free = shutil.disk_usage(base).free
            if best is None or free > best[1] + 5e9:
                best = (base, free)
    return os.path.join(best[0], "mica_pilot")


BIG = pick_big_dir()
os.makedirs(BIG, exist_ok=True)


class Monitor(threading.Thread):
    """Samples GPU memory, GPU load and free RAM every few seconds into a csv file; summary() gives the peaks."""

    def __init__(self, path, every=5):
        super().__init__(daemon=True)
        self.path, self.every, self.stop_flag = path, every, threading.Event()
        self.peak_gpu_mb, self.peak_util, self.min_ram_gb, self.samples = 0, 0, None, 0

    def sample(self):
        gpu_mb, util = "", ""
        if shutil.which("nvidia-smi"):
            out = sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits", timeout=20).splitlines()
            try:
                a, b = [x.strip() for x in out[0].split(",")]
                gpu_mb, util = int(a), int(b)
                self.peak_gpu_mb, self.peak_util = max(self.peak_gpu_mb, gpu_mb), max(self.peak_util, util)
            except (ValueError, IndexError):
                gpu_mb, util = "", ""
        ram = ""
        try:
            with open("/proc/meminfo") as f:
                for line in f:
                    if line.startswith("MemAvailable"):
                        ram = round(int(line.split()[1]) / 1048576, 2)
        except OSError:
            pass
        if ram != "":
            self.min_ram_gb = ram if self.min_ram_gb is None else min(self.min_ram_gb, ram)
        self.samples += 1
        with open(self.path, "a", encoding="utf-8") as f:
            f.write("%s,%s,%s,%s\n" % (datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M:%S"), gpu_mb, util, ram))

    def run(self):
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        with open(self.path, "w", encoding="utf-8") as f:
            f.write("time_utc,gpu_mem_used_mb,gpu_util_pct,ram_available_gb\n")
        while not self.stop_flag.is_set():
            try:
                self.sample()
            except Exception:
                pass
            self.stop_flag.wait(self.every)

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)
        return {"peak_gpu_mb": self.peak_gpu_mb, "peak_gpu_util_pct": self.peak_util, "min_ram_available_gb": self.min_ram_gb, "samples": self.samples}


def check_internet():
    import urllib.request
    bad_required = []
    for name, url, required in NET_HOSTS:
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "mica-benchmark-pilot"}), timeout=20) as r:
                r.read(512)
                log("  reachable: %-9s %s" % (name, r.status))
        except Exception as ex:
            log("  NOT reachable: %-9s %s" % (name, str(ex)[:100]))
            if required:
                bad_required.append(name)
    return bad_required


free_big, free_work = round(shutil.disk_usage(BIG).free / 1e9), round(shutil.disk_usage(WORK).free / 1e9)
cpu_model = sh("lscpu | grep -m1 'Model name'").split(":")[-1].strip()
log("platform %s | python %s | %s | %s CPU cores (%s) | %s GB RAM" % (PLATFORM, platform.python_version(), platform.platform(), os.cpu_count(), cpu_model, meminfo_gb()))
log("large files go to %s (%d GB free); results to %s (%d GB free)" % (BIG, free_big, WORK, free_work))
with open("machine.txt", "w", encoding="utf-8") as f:
    f.write(sh("nvidia-smi") + "\n\n" + sh("lscpu") + "\n\n" + sh("free -h") + "\n\n" + sh("df -h") + "\n\n" + sh("g++ --version | head -1; git --version; uname -a; cat /etc/os-release | head -3") + "\n")
STATE["gpu"] = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") if shutil.which("nvidia-smi") else ""
if STATE["gpu"] and not STATE["gpu"].startswith("ERROR"):
    log("GPU: " + STATE["gpu"].replace("\n", " | "))
elif REQUIRE_GPU:
    fail("this session has no GPU. Kaggle: open Session options on the right, set Accelerator to GPU T4 x2 (or GPU P100), then Run all again. "
         "Colab: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
if STATE["ok"]:
    log("checking the internet:")
    missing = check_internet() if NET_HOSTS else []
    if missing:                                    # a site that the notebook needs (required in NET_HOSTS): stop now, before any hour is spent
        fail("the internet is off, or a site that this notebook needs cannot be reached (%s). Kaggle: open Session options on the right and switch Internet on "
             "(the account must be phone-verified), then Run all again. If Internet is on, the site may be down: run again in a few minutes." % ", ".join(missing))


## Step 2. Write the scripts and the frozen data files

The files below are exact copies of the files in the repository (scripts, the AlphaFold Server job table and the hashes of the 26 predictions,
the entry list, the exact list of Python packages). The last cell of this step checks their hashes.

In [ ]:
import os
for d in ("scripts", "data/af3", "env", "logs", "gpu"):
    os.makedirs(d, exist_ok=True)
print("folders ready")


In [ ]:
%%writefile scripts/nbtools.py
#!/usr/bin/env python3
"""Tools that the Phase 4 notebooks share. They were added after the first MICA pilot, which showed three weaknesses:

1. a program that is silent for an hour (Phenix docking) looked like a frozen notebook: `Heartbeat` prints one line every few minutes
   (elapsed time, load, free RAM, the busiest processes, the last line of the program's own log);
2. the zip with the results was only written by the very last cell, so a stopped session left nothing: `pack` writes the zip in
   one step and the notebooks call it after every stage (a fixed name, replaced each time);
3. the last cell used variables of earlier cells and failed after the kernel was restarted: `recover` needs nothing from the notebook,
   it finds the work folder on disk and packs whatever is there.

Everything the zip should hold is copied into the work folder by the notebook, so that these functions only walk that folder.
Not packed: maps and other large data, files over 60 MB, the AlphaFold Server predictions of the user (they must stay private), and any file
that contains one of the secrets (the Phenix password).

    python scripts/nbtools.py --work <work folder> --out <folder for the zip> --notebook <name>      # the recovery path, by hand
"""
from __future__ import annotations

import argparse
import datetime
import json
import os
import subprocess
import sys
import threading
import time
import zipfile

MAX_FILE = 60 * 1000 * 1000
SKIP_DIRS = {"AF3_results", "af3_upload", "af3_unzipped", "grids", "AF3_encodings", "temp_maps", "native", "downloads", "scripts", "data", "env", "__pycache__"}
SKIP_SUFFIXES = (".map", ".mrc", ".ccp4", ".gz", ".part", ".tmp", ".zip")
WORK_CANDIDATES = ("/kaggle/working/pilot", "/content/pilot")


def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)


def last_line(path, max_chars=160):
    """The last non-empty line of a text file, or "" if there is none or the file cannot be read."""
    try:
        with open(path, "rb") as f:
            f.seek(0, os.SEEK_END)
            size = f.tell()
            f.seek(max(0, size - 8192))
            tail = f.read().decode("utf-8", errors="replace")
    except OSError:
        return ""
    lines = [ln.strip() for ln in tail.splitlines() if ln.strip()]
    return lines[-1][:max_chars] if lines else ""


def machine_load():
    """(load average over 1 minute, free RAM in GB, text about the three busiest processes), each None or "" where it cannot be read."""
    load = free = None
    try:
        with open("/proc/loadavg") as f:
            load = float(f.read().split()[0])
    except (OSError, ValueError, IndexError):
        pass
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable"):
                    free = round(int(line.split()[1]) / 1048576, 1)
    except (OSError, ValueError, IndexError):
        pass
    top = ""
    try:
        r = subprocess.run(["ps", "-eo", "pcpu,etimes,comm", "--sort=-pcpu"], capture_output=True, text=True, timeout=10, errors="replace")
        rows = [ln.split(None, 2) for ln in r.stdout.splitlines()[1:4] if ln.strip()]
        top = ", ".join("%s %s%% (%ss)" % (c[2][:18], c[0], c[1]) for c in rows if len(c) == 3)
    except (OSError, subprocess.SubprocessError):
        pass
    return load, free, top


class Heartbeat(threading.Thread):
    """While a long program runs in the main thread, prints one line every `every` seconds through `log`. `watch` is a list of files whose last
    line is shown (the program's own log); `on_tick` is called after each line (the notebooks use it to refresh the partial zip)."""

    def __init__(self, label, log, watch=(), every=120, on_tick=None):
        super().__init__(daemon=True)
        self.label, self.log, self.watch, self.every, self.on_tick = label, log, list(watch), every, on_tick
        self.stop_flag, self.t0, self.ticks = threading.Event(), time.time(), 0

    def line(self):
        load, free, top = machine_load()
        minutes = (time.time() - self.t0) / 60
        parts = ["[%s] running for %.0f min" % (self.label, minutes) if minutes >= 1 else "[%s] running for %.0f s" % (self.label, time.time() - self.t0)]
        if load is not None:
            parts.append("load %.1f" % load)
        if free is not None:
            parts.append("free RAM %.1f GB" % free)
        if top:
            parts.append("busiest: " + top)
        for path in self.watch:
            if os.path.exists(path):
                age = time.time() - os.path.getmtime(path)
                parts.append("%s: %d bytes, changed %s ago, last line: %s" % (os.path.basename(path), os.path.getsize(path),
                                                                           "%.0f s" % age if age < 120 else "%.0f min" % (age / 60), last_line(path) or "(empty)"))
        return " | ".join(parts)

    def run(self):
        while not self.stop_flag.wait(self.every):
            try:
                self.log(self.line())
                self.ticks += 1
                if self.on_tick:
                    self.on_tick()
            except Exception as ex:                       # a failing heartbeat must never stop the program it watches
                try:
                    self.log("[%s] heartbeat problem: %s" % (self.label, str(ex)[:150]))
                except Exception:
                    pass

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)


def collect(work, skip_dirs=SKIP_DIRS, skip_suffixes=SKIP_SUFFIXES):
    """All files below `work` that may go into the zip, in a fixed order."""
    files = []
    for dirpath, dirs, names in os.walk(work):
        dirs[:] = sorted(d for d in dirs if d not in skip_dirs)
        for n in sorted(names):
            if not n.endswith(skip_suffixes):
                files.append(os.path.join(dirpath, n))
    return files


def pack(work, zip_path, root_name, secrets=(), max_file=MAX_FILE, extra=()):
    """Writes `zip_path` in one step (a temporary file that replaces the old zip at the end, so a stop in the middle never leaves a broken zip).
    Returns (number of files, list of what was left out and why). `extra` are files outside `work` that go in as well."""
    wanted = collect(work) + [p for p in extra if os.path.isfile(p)]
    left_out, tmp = [], zip_path + ".tmp"
    try:
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for path in dict.fromkeys(wanted):
                try:
                    if os.path.getsize(path) > max_file:
                        left_out.append("%s (larger than %d MB)" % (os.path.relpath(path, work), max_file // 1000000))
                        continue
                    with open(path, "rb") as f:
                        data = f.read()
                except OSError as ex:
                    left_out.append("%s (%s)" % (os.path.relpath(path, work), str(ex)[:60]))
                    continue
                if any(s and s.encode() in data for s in secrets):
                    left_out.append("%s (contained the password)" % os.path.relpath(path, work))
                    continue
                inside = os.path.relpath(path, work) if path.startswith(work + os.sep) else os.path.join("outside", path.lstrip(os.sep))
                z.writestr(root_name + "/" + inside, data)
            n = len(z.namelist())
        os.replace(tmp, zip_path)
    except BaseException:
        try:
            os.remove(tmp)
        except OSError:
            pass
        raise
    return n, left_out


def partial_zip_path(out_dir, notebook):
    return os.path.join(out_dir, "%s_partial.zip" % notebook)


def final_pack(work, out_dir, notebook, complete, secrets=(), extra=(), note=""):
    """The zip that is sent to me. complete: True (everything ran), False (something stopped) or None (unknown: recovered after a restart).
    The partial zip is removed once this one exists."""
    stamp = utc_now().strftime("%Y%m%d-%H%M")
    suffix = "" if complete else ("_INCOMPLETE" if complete is False else "_RECOVERED")
    name = "%s_result_%s%s" % (notebook, stamp, suffix)
    path = os.path.join(out_dir, name + ".zip")
    n, left_out = pack(work, path, name, secrets=secrets, extra=extra)
    try:
        os.remove(partial_zip_path(out_dir, notebook))
    except OSError:
        pass
    return path, n, left_out


def find_work(env=None):
    """The work folder of a notebook that ran in this session: the one with a run.log."""
    env = os.environ if env is None else env
    for cand in (env.get("MICA_WORK"),) + WORK_CANDIDATES + (os.path.abspath("pilot"),):
        if cand and os.path.isfile(os.path.join(cand, "run.log")):
            return cand
    return None


def known_secrets(env=None):
    """The Phenix password, if it can be read again (environment, Kaggle or Colab secrets), to check the files for it after a restart."""
    env = os.environ if env is None else env
    found = []
    if env.get("PHENIX_PASSWORD"):
        found.append(env["PHENIX_PASSWORD"].strip())
    for getter in ("kaggle", "colab"):
        try:
            if getter == "kaggle":
                from kaggle_secrets import UserSecretsClient
                v = UserSecretsClient().get_secret("PHENIX_PASSWORD")
            else:
                from google.colab import userdata
                v = userdata.get("PHENIX_PASSWORD")
            if v:
                found.append(v.strip())
        except Exception:
            pass
    return [s for s in dict.fromkeys(found) if len(s) >= 6]


def read_state(work):
    try:
        with open(os.path.join(work, "state.json"), encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def recover(notebook, out_dir=None, env=None, say=print):
    """The last step when nothing from the notebook is left in memory (kernel restarted, cell run alone, ...). Returns the zip path or None."""
    env = os.environ if env is None else env
    work = find_work(env)
    if not work:
        say("RECOVERY: no work folder with a run.log was found (looked in %s). Nothing to pack." % ", ".join(WORK_CANDIDATES))
        return None
    out_dir = out_dir or env.get("MICA_WORK") or os.path.dirname(work)
    state = read_state(work)
    info = {"notebook": notebook, "recovered": True, "recovered_utc": utc_now().isoformat(), "state_json_found": bool(state), "state": state}
    with open(os.path.join(work, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    path, n, left_out = final_pack(work, out_dir, notebook, complete=None, secrets=known_secrets(env))
    say("RECOVERY: zip written from the files that are on disk: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        say("left out of the zip: %s" % "; ".join(left_out))
    say("If a notebook cell stopped with an error, send me this zip and the text of the error.")
    return path


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--work", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--notebook", required=True)
    args = p.parse_args(argv)
    path, n, left_out = final_pack(args.work, args.out, args.notebook, complete=None)
    print("%s (%d files)" % (path, n))
    for x in left_out:
        print("left out:", x)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/mica_inputs.py
#!/usr/bin/env python3
"""Phase 4: the input folder that MICA expects, for one entry of the frozen list.

MICA reads (README, "Running on New Dataset", and utils/process_AF3_results.py):

    <root>/<PDB ID>/
        <pdb id>.fasta                         one record per unique chain sequence, header  >26DK_1|Chains A, B|...
        emd_<n>.map                            (put there by the notebook)
        AF3_results/<pdb id>_<k>/fold_<pdb id>_<k>_model_0.cif

The header must start with the PDB ID, one underscore, the number of the sequence, then "|Chains ..." (the number of chains is
the number of copies of that sequence in the entry). The third part of the file name (split at "_") must be the number k, and
the PDB ID of the first header must be the one of the files, or MICA finds no protein for the file and drops it silently.
AlphaFold Server returned the files as fold_<job name in lower case>_model_0.cif with the job names of data/af3/af3_jobs.csv
(for example fold_26dk_e1_model_0.cif); this script renames E1, E2, ... to 1, 2, ... and writes the FASTA file from the same table, so the
sequence MICA reads is the sequence that AlphaFold Server was given (letters outside the 20 amino acids were replaced by G there).

    python scripts/mica_inputs.py --jobs data/af3/af3_jobs.csv --models <folder with the 26 files> --pdb 26DK --out <root>
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import shutil
import sys

SCRIPT_VERSION = "1.0"
CHAIN_LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"


def read_jobs(path, pdb_id):
    """The rows of af3_jobs.csv for one entry, in the order of the entities (E1, E2, ...)."""
    with open(path, newline="", encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r["pdb_id"].upper() == pdb_id.upper()]
    if not rows:
        raise ValueError("%s: no job for %s" % (path, pdb_id))
    rows.sort(key=lambda r: int(r["entity"].lstrip("Ee")))
    numbers = [int(r["entity"].lstrip("Ee")) for r in rows]
    if numbers != list(range(1, len(rows) + 1)):
        raise ValueError("%s: the entities of %s are %s, expected 1 to %d without gaps" % (path, pdb_id, numbers, len(rows)))
    return rows


def find_model(models_dir, job_name):
    """Path of fold_<job name in lower case>_model_0.cif somewhere under models_dir (case does not matter)."""
    want = "fold_%s_model_0.cif" % job_name.lower()
    hits = []
    for root, _dirs, files in os.walk(models_dir):
        hits += [os.path.join(root, f) for f in files if f.lower() == want]
    if not hits:
        raise ValueError("no file %s under %s" % (want, models_dir))
    if len(hits) > 1:
        raise ValueError("%s occurs %d times under %s" % (want, len(hits), models_dir))
    return hits[0]


def mica_fasta(rows):
    """FASTA text with MICA's header format. The chain letters run through the entry (A, B for the first sequence, C, D for the next, ...)."""
    total = sum(int(r["copies_in_entry"]) for r in rows)
    if total > len(CHAIN_LETTERS):
        raise ValueError("%s has %d chains; MICA writes PDB files and cannot name more than %d" % (rows[0]["pdb_id"], total, len(CHAIN_LETTERS)))
    out, used = [], 0
    for k, r in enumerate(rows, 1):
        copies = int(r["copies_in_entry"])
        if copies < 1:
            raise ValueError("%s: copies_in_entry is %s" % (r["job_name"], r["copies_in_entry"]))
        letters = ", ".join(CHAIN_LETTERS[used:used + copies])
        used += copies
        pdb = r["pdb_id"].upper()
        if "_" in pdb or "|" in pdb:
            raise ValueError("the PDB ID %r contains _ or |" % pdb)
        out.append(">%s_%d|Chains %s|%s sequence %d|" % (pdb, k, letters, pdb, k))
        out.append(r["sequence_submitted"])
    return "\n".join(out) + "\n"


def build(jobs_csv, models_dir, pdb_id, out_root):
    """Writes <out_root>/<PDB ID>/ with the FASTA file and AF3_results; returns a dict that says what was written."""
    rows = read_jobs(jobs_csv, pdb_id)
    pdb = rows[0]["pdb_id"].upper()
    low = pdb.lower()
    folder = os.path.abspath(os.path.join(out_root, pdb))
    results = os.path.join(folder, "AF3_results")
    # MICA derives its other folders with AF3_results_path.replace("results", ...): the word must occur only once, in the last part
    if "results" in os.path.dirname(results):
        raise ValueError("the path %s contains the word 'results' before AF3_results; MICA would write into the wrong folders. Use another root folder" % folder)
    os.makedirs(results, exist_ok=True)
    fasta_path = os.path.join(folder, low + ".fasta")
    with open(fasta_path, "w", encoding="utf-8") as f:
        f.write(mica_fasta(rows))
    files = []
    for k, r in enumerate(rows, 1):
        src = find_model(models_dir, r["job_name"])
        d = os.path.join(results, "%s_%d" % (low, k))
        os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, "fold_%s_%d_model_0.cif" % (low, k))
        shutil.copyfile(src, dst)
        files.append({"job_name": r["job_name"], "source": os.path.basename(src), "target": os.path.relpath(dst, folder)})
    return {"pdb_id": pdb, "folder": folder, "fasta": fasta_path, "af3_results": results, "files": files}


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--jobs", required=True, help="data/af3/af3_jobs.csv")
    p.add_argument("--models", required=True, help="folder with the fold_*_model_0.cif files from AlphaFold Server")
    p.add_argument("--pdb", required=True, help="PDB ID of the entry, for example 26DK")
    p.add_argument("--out", required=True, help="root folder; the entry gets <out>/<PDB ID>/")
    args = p.parse_args(argv)
    try:
        info = build(args.jobs, args.models, args.pdb, args.out)
    except (ValueError, OSError, KeyError) as e:
        print("ERROR: %s" % e, file=sys.stderr)
        return 2
    print(json.dumps(info, indent=1))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/evaluate_models.py
#!/usr/bin/env python3
"""Phase 4: scores one predicted model against the deposited model (docs/SPEC.md, decision 7), version 0.1 for the pilot.

What it computes with US-align (protein chains only, `-mol protein`, tabular output `-outfmt 2`; the prediction is structure 1 and the
deposited model is structure 2, so TM2 is the TM-score normalised by the length of the deposited model, the one that the US-align
output recommends):

    tm_score            TM2, normalised by the deposited model
    tm_score_pred_norm  TM1, normalised by the prediction (for reference only)
    ca_rmsd             RMSD over the aligned residues
    aligned_ca_length   number of aligned residues (Lali)
    seq_identity        identical residues among the aligned ones (IDali)
    pred_residues       protein residues in the prediction
    native_residues     protein residues in the deposited model (US-align L2)
    coverage            built residues / deposited residues; the deposited count is `modeled_residues` of data/entries_2026.csv

The Phenix part (Calpha match and sequence match from phenix.chain_comparison, and with them the Calpha quality score) is not here yet:
the pilot saves the raw Phenix output so that the exact command and the output lines can be fixed before the scripts are trusted.

    python scripts/evaluate_models.py --usalign <path to USalign> --pred model.pdb --native 26DK.cif --modeled-residues 191
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys

SCRIPT_VERSION = "0.1"
TABLE_COLUMNS = ["PDBchain1", "PDBchain2", "TM1", "TM2", "RMSD", "ID1", "ID2", "IDali", "L1", "L2", "Lali"]


def parse_usalign_tabular(text):
    """The rows of `USalign -outfmt 2`: a list of dicts with numbers as floats (L1, L2 and Lali as integers)."""
    rows = []
    for line in text.splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) != len(TABLE_COLUMNS):
            raise ValueError("a US-align line has %d fields, %d expected: %r" % (len(parts), len(TABLE_COLUMNS), line[:200]))
        row = dict(zip(TABLE_COLUMNS, parts))
        for k in ("TM1", "TM2", "RMSD", "ID1", "ID2", "IDali"):
            row[k] = float(row[k])
        for k in ("L1", "L2", "Lali"):
            row[k] = int(row[k])
        rows.append(row)
    return rows


def run_usalign(usalign, pred, native, timeout=900):
    """US-align of the prediction onto the deposited model; returns the one table row, or raises RuntimeError with the reason."""
    cmd = [usalign, pred, native, "-mol", "protein", "-outfmt", "2"]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, errors="replace")
    except subprocess.TimeoutExpired:
        raise RuntimeError("US-align did not finish within %d seconds" % timeout)
    except OSError as e:
        raise RuntimeError("US-align could not be started: %s" % e)
    if r.returncode != 0:
        raise RuntimeError("US-align stopped with exit code %d: %s" % (r.returncode, (r.stderr or r.stdout).strip()[-300:]))
    rows = parse_usalign_tabular(r.stdout)
    if len(rows) != 1:
        raise RuntimeError("US-align gave %d result lines, 1 expected. Output: %s" % (len(rows), r.stdout.strip()[-300:]))
    return rows[0]


def evaluate(usalign, pred, native, modeled_residues):
    """The numbers of one model. status is "ok", or the reason why no alignment exists (then the numbers are empty)."""
    out = {"status": "ok", "note": "", "tm_score": "", "tm_score_pred_norm": "", "ca_rmsd": "", "aligned_ca_length": "", "seq_identity": "",
           "pred_residues": "", "native_residues": "", "coverage": ""}
    try:
        row = run_usalign(usalign, pred, native)
    except (RuntimeError, ValueError) as e:
        out.update(status="no_alignment", note=str(e)[:300])
        return out
    out.update(tm_score=row["TM2"], tm_score_pred_norm=row["TM1"], ca_rmsd=row["RMSD"], aligned_ca_length=row["Lali"], seq_identity=row["IDali"],
               pred_residues=row["L1"], native_residues=row["L2"])
    if modeled_residues:
        out["coverage"] = round(row["L1"] / float(modeled_residues), 4)
    return out


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--usalign", required=True, help="path to the USalign program")
    p.add_argument("--pred", required=True, help="predicted model (pdb or cif)")
    p.add_argument("--native", required=True, help="deposited model (pdb or cif)")
    p.add_argument("--modeled-residues", type=int, default=0, help="modeled_residues of the entry in data/entries_2026.csv")
    p.add_argument("--out", help="write the numbers as json")
    args = p.parse_args(argv)
    res = evaluate(args.usalign, args.pred, args.native, args.modeled_residues)
    print(json.dumps(res, indent=1))
    if args.out:
        with open(args.out, "w", encoding="utf-8") as f:
            json.dump(res, f, indent=1)
    return 0 if res["status"] == "ok" else 2


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile data/af3/af3_jobs.csv
job_no,job_name,pdb_id,entity,stratum,seed_rank,length,copies_in_entry,sequence_submitted,substitution
1,9T5K_E1,9T5K,E1,le3.0,1,453,1,MHHHHHHGKPIPNPLLGLDSTENLYFQGIDPFTMFSTLKNETEKFREYQANLGRQGKPLASTATLTTKIIVYNPSNKSPALRWEITKFAMRLIWSPAASHSVKVGAALTLLSAHAENPGAMIRSLVNDPDIEVVITDISEFDHGVPRLERRGEKAEQQMDSYRRILDRAPQENLFYNPEVDDLEILDSGTFLFAIATVLAQVWILVAKAVTAPDTAEESENKRWAKYVQQKRVNPDYLVSNRWITAMRSLISIDLSVRKYMVEILIEVKKSGVARGRLNEMIADIGNYIEETGMAGFFLTIKYGLEMKFPVIVINEFQADLLTLQTLMRTYMDLGPRAPYMVLLEDSIQTKFAPGNYPLLWSFAMGVGTTLDRSMGALNINRSYLEPIYFKLGQNAARKNAGSIDRKLAEELGLTQEQANEIKEMMQEVTTQRHETNVQAREGKFNVAAGGIE,
2,22FX_E1,22FX,E1,le3.0,2,182,1,MTTASPSQVRQNYHQDAEAAINRQINLELYASYVYLSMSCYFDRDDVALKNFAKYFLHQSHEEREHAEKLMKLQNQRGGRIFLQDIKKPDRDDWESGLNAMECALHLEKSVNQSLLELHKLATDKNDPHLCDFIETYYLSEQVKSIKELGDHVTNLRKMGAPEAGMAEYLFDKHTLGHGDES,
3,9JKK_E1,9JKK,E1,le3.0,3,620,1,MSKSKCSVGLMSSVVAPAKEPNAVGPKEVELILVKEQNGVQLTSSTLTNPRQSPVEAQDRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV,
4,9TKA_E1,9TKA,E1,le3.0,4,293,1,MNPASDGGTSESIFDLDYASWGIRSTLMVAGFVFYLGVFVVCHQLSSSLNATYRSLVAREKVFWDLAATRAVFGVQSTAAGLWALLGDPVLHADKARGQQNWCWFHITTATGFFCFENVAVHLSNLIFRTFDLFLVIHHLFAFLGFLGCLVNLQAGHYLAMTTLLLEMSTPFTCVSWMLLKAGWSESLFWKLNQWLMIHMFHCRMVLTYHMWWVCFWHWDGLVSSLYLPHLTLFLVGLALLTLIINPYWTHKKTQQLLNPVDWNFAQPEAKSRPEGNGQLLRKKRPDAAIEGR,
5,9WQ6_E1,9WQ6,E1,gt3.0_le4.0,4,339,1,LSLLHLQEKNWSALLTAVVIILTIAGNILVIMAVSLEKKLQNATNYFLMSLAIADMLLGFLVMPVSMLTILYGYRWPLPSKLCAVWIYLDVLFSTASIMHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMPIPVFGLQDDSKVFKEGSCLLADDNFVLIGSFVSFFIPLTIMVITYFLTIKSLQKEATLCVSDLGTRAKLASFSFLPQSSLSSEKLFQRSIHREPGSYTGRRTMQSISNEQKACKVLGIVFFLFVVMWCPFFITNIMAVICKESCNEDVIGALLNVFVWIGYLSSAVNPLVYTLFNKTYRSAFSRYIQCQYKENKK,
6,9WFL_E1,9WFL,E1,le3.0,5,697,1,SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRPF,
7,9WFL_E2,9WFL,E2,le3.0,5,29,1,RVFLTGEKANSILKRYPRANGFFEEIRQG,
8,9V4D_E1,9V4D,E1,gt3.0_le4.0,5,393,1,MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSGNDSVSYEYGDYSDLSDRPVDCLDGACLAIDPLRVAPLPLYAAIFLVGVPGNAMVAWVAGKVARRRVGATWLLHLAVADLLCCLSLPILAVPIARGGHWPYGAVGCRALPSIILLTMYASVLLLAALSADLCFLALGPAWWSTVQRACGVQVACGAAWTLALLLTVPSAIYRRLHQEHFPARLQCVVDYGGSSSTENAVTAIRFLFGFLGPLVAVASCHSALLCWAARRCRPLGTAIVVGFFVCWAPYHLLGLVLTVAAPNSALLARALRAEPLIVGLALAHSCLNPMLFLYFGRAQLRRSLPAACHWALRESQGQDESVDSKKSTSHDLVSEMEV,
9,9V4D_E2,9V4D,E2,gt3.0_le4.0,5,9,1,GYKPLGLAY,1 residue(s) X replaced by G
10,9VI4_E1,9VI4,E1,le3.0,6,516,1,VGNASGNWHCDSTWLGDRVITTSTRTWALPTYNNHLYKQISSASTGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTTNDGVTTIANNLTSTVQVFSDSEYQLPYVLGSAHQGCLPPFPADVFMIPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLNRTQNQSGSAQNKDLLFSRGSPAGMSVQPKNWLPGPCYRQQRVSKTKTDNNNSNFTWTGASKYNLNGRESIINPGTAMASHKDDKDKFFPMSGVMIFGKESAGASNTALDNVMITDEEEIKATNPVATERFGTVATNHQSVNYEEETGDVHVMGALPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPPAEFSATKFASFITQYSTGQVSVEIEWELQKENSKRWNPEVQYTSNYAKSANVDFTVDNNGLYTEPRPIGTRYLTRPL,
11,9ZLH_E1,9ZLH,E1,le3.0,7,206,1,DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEP,
12,9OTA_E1,9OTA,E1,gt3.0_le4.0,7,270,2,GALLAGALAAYAAYLVLGALLVARLEGPHEARLRAELETLRAQLLQRSPCVAAPALDAFVERVLAAGRLGRVVLANASGSANASDPAWDFASALFFASTLITTVGYGYTTPLTDAGKAFSIAFALLGVPTTMLLLTASAQRLSLLLTHVPLSWLSMRWGWDPRRAACWHLVALLGVVVTVCFLVPAVIFAHLEEAWSFLDAFYFCFISLSTIGLGDYVPGEAPGQPYRALYKVLVTVYLFLGLVAMVLVLQTFRHVSDLHGLTELILLPP,
13,9VUH_E1,9VUH,E1,le3.0,8,332,1,MADYKDDDDKSGRGAMGALQGCRSVEEFQCLNRIEEGTYGVVYRAKDKKTDEIVALKRLKMEKEKEGFPITSLREINTILKAQHPNIVTVREIVVGSNMDKIYIVMNYVEHDLKSLMETMKQPFLPGEVKTLMIQLLRGVKHLHDNWILHRDLKTSNLLLSHAGILKVGDFGLAREYGSPLKAYTPVVVTLWYRAPELLLGAKEYSTAVDMWSVGCIFGELLTQKPLFPGKSEIDQINKVFKDLGTPSEKIWPGYSELPAVKKMTFSEHPYNNLRKRFGALLSDQGFDLMNKFLTYFPGRRISAEDGLKHEYFRETPLPIDPSMFPKLVEKY,
14,9VUH_E2,9VUH,E2,le3.0,8,365,1,MASGPHSTATAAAAASSAAPSAGGSSSGTTTTTTTTTGGILIGDRLYSEVSLTIDHSLIPEERLSPTPSMQDGLDLPSETDLRILGCELIQAAGILLRLPQVAMATGQVLFHRFFYSKSFVKHSFEIVAMACINLASKIEEAPRRIRDVINVFHHLRQLRGKRTPSPLILDQNYINTKNQVIKAERRVLKELGFCVHVKHPHKIIVMYLQVLECERNQTLVQTAWNYMNDSLRTNVFVRFQPETIACACIYLAARALQIPLPTRPHWFLLFGTTEEEIQEICIETLRLYTRKKPNYELLEKEVEKRKVALQEAKLKAKGLNPDGRIPGLIWSHPQFEKGGGSGGGGGSGGGSGGSSAWSHPQFEK,
15,9VUH_E3,9VUH,E3,le3.0,8,318,1,MGSENLYFQGMAGKKNVLSSLAVYAEDSEPESDGEAGIEAVGSTAEEKGGLVSDAYGEDDFSRLGGDEDGYEEEEDENSRQSEDDDSETEKPEADDPKDNTEAEKRDPQELVASFSERVRNMSPDEIKIPPEPPGRCSNHLQDKIQKLYERKIKEGMDMNYIIQRKKEFRNPSIYEKLIQFCAIDELGTNYPKDMFDPHGWSEDSYYEALAKAQKIEMDKLEKAKKERTKIEFVTGTKKGTTTNATSTTTTTASTAVADAQKRKSKWDSAIPVTTIAQPTILTTTATLPAVVTVTTSASGSKTTVISAVGTIVKNAKQ,
16,9KQM_E1,9KQM,E1,gt3.0_le4.0,8,637,1,MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLQEPRRQRRLVLVIVCVALLLDNMLYMVIVPIVPDYIAHMRGGGEGPTRTPEVWEPTLPLPTPANASAYTANTSASPTAAWPAGSALRPRYPTESEDVKIGVLFASKAILQLLVNPLSGPFIDRMSYDVPLLIGLGVMFASTVLFAFAEDYATLFAARSLQGLGSAFADTSGIAMIADKYPEEPERSRALGVALAFISFGSLVAPPFGGILYEFAGKRVPFLVLAAVSLFDALLLLAVAKPFSAAARARANLPVGTPIHRLMLDPYIAVVAGALTTCNIPLAFLEPTIATWMKHTMAASEWEMGMAWLPAFVPHVLGVYLTVRLAARYPHLQWLYGALGLAVIGASSCIVPACRSFAPLVVSLCGLCFGIALVDTALLPTLAFLVDVRHVSVYGSVYAIADISYSVAYALGPIVAGHIVHSLGFEQLSLGMGLANLLYAPVLLLLRNVGLLTRSRSERDVLLDEPPQGLYDAVRLRERPVSGQDGEPRSPPGPFDACEDDYNYYYTRSHHHHHHHHHHGSVEDYKDDDDK,
17,9WME_E1,9WME,E1,gt3.0_le4.0,9,453,1,MTPSISKDAPIKGSITISKKGATFTAYKLLDAIKSGDAYEYSVNSDLKDFFNNSNYGSYSQESIQKLNGEQVKEFAINLHKYILENKKSGQELKDGQKNTVDLGYYLVTETSSDSEGAAVASTPIIVSVPQVSGDSWNYDVTINPKDNTPILEKNIVKENQRVKTSSENIGDVVKYEVKASIPVYQKNAQNIMYKFTDTMSKGLTYDEKTGFKVTSGDKVFAKDTDYTVDVKKQEDGSTVITINFVYENIKAYAETGITLNYQATLNKDAVISNKENLGNTNNIQLDYTNNPHVKDSYKKLTDKVTTYTFGFGITKVDSELNSKLLQGAEFSVKDAGGKIVAKYTYDEKGQVVYLSGNGVTNSKGITTFLGLKEGKYFITEEVAPSGYSLLKNPVEVTITANKDESGNYTGAATIEISNGNKAGQIINDISEKDGNILFNVQIENHAHHHHHH,
18,9WME_E2,9WME,E2,gt3.0_le4.0,9,657,1,MEETKSVQNDGAVEITSTTFESNTVAKGISNNLRIDYKILNKDKLKDGDKIVISLPDIFKDIEPKCHDQHFKDFDVKDGVVTLTFNENVEKAVTGYMIIRFVGNSNIRKGVSYPVSIDLNGKPSTVYITGEEYSNSSSGAQYPLMYKTADLPTAATDKEQGREYYGEIVDRNKPIKYFVEINLGDGVNPNTRSYLSNADFFDNIPKGMALDVNSICIKRMGYYDERSSDVTKDFWESNRIKADTKHLEINFGDIRYERYTVIYETRITSTESGYLNDAKLYYDDKELPSKHYSKLSKDAGALNVYKYVDKTKVKNNLNDQKIKYDIKFDSYGYFFKDTLNIIDKLDPRLSDIKITATDQFITDFDENTKELVIKNSNGDIDAKKPAYITIEASMKNVGPGEVVKNIAYVNGNPTNEVSTRKNPIVEIIKVSKDDVESNLLEGAIFKLTTKDGKTVKDVYNKGVKTFTTSSEGSIRFELPNGDYKLEEIKAPNGYKLDQTPIEFTVNDESKIVKVVAKDDPLPTTCNLVINKINEKEIPILGAKFKLFEESNPKKVLKFSSQKNNYELNQNGVGKLTPSGKNASFKINNLHYGNYILKEVQAPKGYKLSDDIYITLGFEESFYRVGKQGEKIILNKNTETNTYNISVENVPRIILPET,
19,9OC0_E1,9OC0,E1,le3.0,11,847,2,MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ,
20,9OC0_E2,9OC0,E2,le3.0,11,862,2,GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG,
21,13BD_E1,13BD,E1,le3.0,12,1271,2,MFVFLVLLLLVSSQCVNLTTTTQLPLAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHVISGTNGTKRFDNPVLPFNDGVYFASIEKSNITRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFNNKSWMESESRVYSSAKNCTFEYVFQPFLMDLEGKQGNFTNLREFVFKNIDGYFKIYSKHTPIIVRDLPQGFSALEPLVDLPIGINITRFQTLLHRSYLTSGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESVVRFPNITNLCPFYEVFNATRFPSVYAWNRKRISNCVADYSVLYNFAPFFTFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVISWNSNRLDSKADGNYNYWYRLFRKSKLKPFERDISTEIYQAGNNPCNGVKGFNCYFPLQSYSFRPTYGVGYQPYRVVVLSFELLHAPATVCGPKQSTNLVKNKCVNFNFNGLTGTGVLTDSNKKFLPFQQFGRDIADTTDAVRDPQTLDILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVPVAIRADQLTPTWRVYSTGSKGFQTRAGCLIGAKYVNNSYECDIPIGAGICASYQTQTRSRGSASSVDSQSIIAYTMSLGAENLVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPITYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICKQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSRDVDLGDISGINASVVNIQKEIERLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEK,
22,9VBD_E1,9VBD,E1,gt3.0_le4.0,15,893,1,LTNGLDASALNALKSEWTTPPDGWEGSDPCGTNWVGITCQNDRVVSISLGNLDLEGKLPADISFLSELRILDLSYNPKLSGPLPPNIGNLGKLRNLILVGCSFSGQIPESIGTLKELIYLSLNLNKFSGTIPPSIGLLSKLYWFDIADNQIEGELPVSNGTSAPGLDMLLQTKHFHFGKNKLSGNIPKELFSSNMSLIHVLFDGNQFTGEIPETLSLVKTLTVLRLDRNKLIGDIPSYLNNLTNLNELYLANNRFTGTLPNLTSLTSLYTLDVSNNTLDFSPIPSWISSLPSLSTLRMEGIQLNGPIPISFFSPPQLQTVILKRNSIVESLDFGTDVSSQLEFVDLQYNEITDYKPSANKVLQVILANNPVCLEAGNGPSYCSAIQHNTSFSTLPTNCSPCEPGMEASPTCRCAYPFMGTLYFRSPSFSGLFNSTNFSILQKAIADFFKKFNYPVDSVGVRNIRENPTDHQLLIDLLVFPLGRESFNQTGMSLVGFAFSNQTYKPPPIFGPYIFKADLYKQFSGSAMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQT,
23,9RWS_E1,9RWS,E1,gt3.0_le4.0,16,551,1,MQPQPSATPDRKQDNKNKNYEKAGLLGFIGLLLVVAVYVAFTGNDVNWGALVFMFISYAIIFYIGSVTAGKKSDSASDMMVAGRSMPLWVAMFTMTATWVGGGYIAGTAETTFASGVVWAQAPWGYGLSLIIGGIFYARKMRRYEFTTMLDPLEVRFGKKVAGVLYLPALLGELFWSAAILVALGTTFGLILGLDFTTSIIISAIIAVAYTFVGGMWSVALTDVAQIIMIIIGMFLVVPFALSEVGGLGQAWGAYKEGMAGFTNLLPPLDGWNHPDWGNYYWNWWDYALLLIFGGIPWQVYFQRVLSAKNEKTAMWLSIAAGFLCIILAVPAVMIGVAGFSADWASYGIEGPGAASEILAYVVNYMSPYLVAIVALGAIAAAVMSSMDSSILSASSMAAWNIYRPLVKPKANGRDIKRTIRVSIILIGLAATIVALNIDSVYTLWYLCADLVYCMLFPQLTTALFDKKANTYGAIAGLSVSLFLRLGGGEPALGLPAFLPYPMIEDGVVLFPFRTLAMVSGLLTIIVVSRLTQKSCPPQPLRKLKTEMQDE,
24,9WTX_E1,9WTX,E1,gt3.0_le4.0,17,551,1,MRDYDEVIAFLGEWGPFQRLIFFLLSASIIPNGFNGMSVVFLAGTPEHRCRVPDAANLSSAWRNNSVPLRLRDGREVPHSCSRYRLATIANFSALGLEPGRDVDLGQLEQESCLDGWEFSQDVYLSTVVTEWNLVCEDNWKVPLTTSLFFVGVLLGSFVSGQLSDRFGRKNVLFATMAVQTGFSFLQIFSISWEMFTVLFVIVGMGQISNYVVAFILGTEILGKSVRIIFSTLGVCTFFAVGYMLLPLFAYFIRDWRMLLLALTVPGVLCVPLWWFIPESPRWLISQRRFREAEDIIQKAAKMNNIAVPAVIFDSVEELNPLKQQKAFILDLFRTRNIAIMTIMSLLLWMLTSVGYFALSLDAPNLHGDAYLNCFLSALIEIPAYITAWLLLRTLPRRYIIAAVLFWGGGVLLFIQLVPVDYYFLSIGLVMLGKFGITSAFSMLYVFTAELYPTLVRNMAVGVTSTASRVGSIIAPYFVYLGAYNRMLPYIVMGSLTVLIGILTLFFPESLGMTLPETLEQMQKVKWFRSGKKTRDSMETEENPKVLITAF,
25,9Z24_E1,9Z24,E1,gt3.0_le4.0,18,1214,1,MVVPEKEQSWIPKIFKKKTCTTFIVDSTDPGGTLCQCGRPRTAHPAVAMEDAFGAAVVTVWDSDAHTTEKPTDAYGELDFTGAGRKHSNFLRLSDRTDPAAVYSLVTRTWGFRAPNLVVSVLGGSGGPVLQTWLQDLLRRGLVRAAQSTGAWIVTGGLHTGIGRHVGVAVRDHQMASTGGTKVVAMGVAPWGVVRNRDTLINPKGSFPARYRWRGDPEDGVQFPLDYNYSAFFLVDDGTHGCLGGENRFRLRLESYISQQKTGVGGTGIDIPVLLLLIDGDEKMLTRIENATQAQLPCLLVAGSGGAADCLAETLEDTLAPGSGGARQGEARDRIRRFFPKGDLEVLQAQVERIMTRKELLTVYSSEDGSEEFETIVLKALVKACGSSEASAYLDELRLAVAWNRVDIAQSELFRGDIQWRSFHLEASLMDALLNDRPEFVRLLISHGLSLGHFLTPMRLAQLYSAAPSNSLIRNLLDQASHSAGTKAPALKGGAAELRPPDVGHVLRMLLGKMCAPRYPSGGAWDPHPGQGFGESMYLLSDKATSPLSLDAGLGQAPWSDLLLWALLLNRAQMAMYFWEMGSNAVSSALGACLLLRVMARLEPDAEEAARRKDLAFKFEGMGVDLFGECYRSSEVRAARLLLRRCPLWGDATCLQLAMQADARAFFAQDGVQSLLTQKWWGDMASTTPIWALVLAFFCPPLIYTRLITFRKSEEEPTREELEFDMDSVINGEGPVGTADPAEKTPLGVPRQSGRPGCCGGRCGGRRCLRRWFHFWGAPVTIFMGNVVSYLLFLLLFSRVLLVDFQPAPPGSLELLLYFWAFTLLCEELRQGLSGGGGSLASGGPGPGHASLSQRLRLYLADSWNQCDLVALTCFLLGVGCRLTPGLYHLGRTVLCIDFMVFTVRLLHIFTVNKQLGPKIVIVSKMMKDVFFFLFFLGVWLVAYGVATEGLLRPRDSDFPSILRRVFYRPYLQIFGQIPQEDMDVALMEHSNCSSEPGFWAHPPGAQAGTCVSQYANWLVVLLLVIFLLVANILLVNLLIAMFSYTFGKVQGNSDLYWKAQRYRLIREFHSRPALAPPFIVISHLRLLLRQLCRRPRSPQPSSPALEHFRVYLSKEAERKLLTWESVHKENFLLARARDKRESDSERLKRTSQKVDLALKQLGHIREYEQRLKVLEREVQQCSRVLGWVAEALSRSALLPPGGPPPPDLPGSKD,
26,26DK_E1,26DK,E1,gt3.0_le4.0,19,197,1,MTCTIDLQILQGKTFEFAFRWADPIKLYKPITGATATAPVVLTVPGHGLSDGWPFEVQSVKAPPELNTQPGQFHLAKVLTPDSLELNDVNGIGFKAFGGAGTIVFYTPADLTGLAARFAIRRTPGDADALLTGNTTDGRVVIDIPTSTISMLIGADVTATLDWNKALYDLELYDPTDVSVVYPVGSGRVTVTAEVAK,


In [ ]:
%%writefile data/af3/af3_files_sha256.txt
0c476efceb778175e74194bb9278507c40fd356717aa2df1cc080dbd550fbb50  fold_9t5k_e1_model_0.cif
d527277831a49730b21307671ee991581cc92159af251d39ab05223853ca55a4  fold_22fx_e1_model_0.cif
e7c802f7d0096e007cb302ff90057fb5f4ed4a90239613b5e9001fe71eeb0a00  fold_9jkk_e1_model_0.cif
5c044c1e79476fe9aef40bb36e91e5f8d9dba5db52e7f113b64783de5a3d9c3d  fold_9tka_e1_model_0.cif
0d615fa596e4f18592f338d49db67a1bd5cf972ac11cfa5991849282549e4701  fold_9wq6_e1_model_0.cif
a641a2fe255412290e1a9055a299e5fe3e7fec7721d2c1e8019ea47e2f4740ee  fold_9wfl_e1_model_0.cif
92f7c374227083b7887daf817e3a5b4794838834bc0c0a4b349f7a9dd898cde1  fold_9wfl_e2_model_0.cif
8a39e8ca50c3734e747413423d3c6f6b2f01a8478d0f70a5d4fbb9ea618be606  fold_9v4d_e1_model_0.cif
657483d66b118a4d6e52ef559a8f202511ffe088a07744eddfcad7dd769a32c7  fold_9v4d_e2_model_0.cif
e6612d7820592e9594cd4252a4ca632d0dd0d0ca6d044876f7ecc5464b5c627a  fold_9vi4_e1_model_0.cif
d3e403fed3ab69c0d5ab37c635821cdf1c63d6c5ee92e315864b35620c9b0d13  fold_9zlh_e1_model_0.cif
35a8509957cc26c2c4d1c2c1d43a9c9919d9da3fb81c785e62dbac37568bc6a6  fold_9ota_e1_model_0.cif
4ff4ee5245115c5ceeaa8675ce177fe2b4eb2e6d11e450e717ee301e35aeb639  fold_9vuh_e1_model_0.cif
7db05c0b4f61ec68dc52166f592a9035512deea665c7e950d5260f73b192c5f5  fold_9vuh_e2_model_0.cif
29226dfc526e553e7d9e624a24ab55d3cb43f43c1c0ddef5e19509c9f5b10216  fold_9vuh_e3_model_0.cif
753663c977ac53243460e8d000965a1b85a7b640123590fe2f0d3aa6713f65ae  fold_9kqm_e1_model_0.cif
456c5f8d1253c1b5da60387870ad303ec289f2cfbfa87ce360e4db9c650581f6  fold_9wme_e1_model_0.cif
94aae35fe8a0b8d4e1e8551f07ab225d2b955f65bba450e1f58b00fb0f5c7324  fold_9wme_e2_model_0.cif
ba3676e2a9be6065dea72a1412bb0fa112740bc0146991713f7cc05d150e622a  fold_9oc0_e1_model_0.cif
cd88f367aa02422caf019d9a39e609ec9adb9082a5571cb30fd1e10874bf639f  fold_9oc0_e2_model_0.cif
e2c5250de4a9d1d198b41c44f0d4d5c1fe7d0412634c43ba79a4372b07248933  fold_13bd_e1_model_0.cif
adab201f4897423c415f43e63323dc30dd751534435cbd97bd835693a2c6e9d5  fold_9vbd_e1_model_0.cif
49ecef225e7c5ded86767bc3d7029e1b29288d950cb3ab4486809ffc7a74f895  fold_9rws_e1_model_0.cif
f183f63d28d4e43d3b1855e6dcb3e2d4c6fc55613dd814c8125c3d1c7ac7950c  fold_9wtx_e1_model_0.cif
63bb6cd913d575b1d143fe48ca68997b507f2d284e1e3a68b630645eb14f034c  fold_9z24_e1_model_0.cif
a1977ebd9d6a90cb0ba3b20631ba084ad4bcffc2e7dc84c897b628ff2cfa9d32  fold_26dk_e1_model_0.cif


In [ ]:
%%writefile data/entries_2026.csv
emdb_id,pdb_id,emdb_release_date,pdb_release_date,resolution_A,stratum,modeled_residues,unique_chain_seqs,unk_fraction,cc_mask,cluster_id,seed_rank,max_id_to_pre2026_pdb,cluster_size,longest_seq_len,n_chains,map_size_mb,cif_url,map_url
EMD-55586,9T5K,2026-04-22,2026-04-22,2.6,le3.0,381,1,0.0,0.8200,9T5K,1,0.614,1,453,1,250.0,https://files.rcsb.org/download/9T5K.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55586/map/emd_55586.map.gz
EMD-68251,22FX,2026-03-04,2026-03-04,1.24,le3.0,173,1,0.0,0.7388,22FX,2,1.000,1,182,1,75.2,https://files.rcsb.org/download/22FX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68251/map/emd_68251.map.gz
EMD-61564,9JKK,2026-04-08,2026-04-08,3.0,le3.0,530,1,0.0,0.8420,9W97,3,1.000,39,620,1,65.5,https://files.rcsb.org/download/9JKK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61564/map/emd_61564.map.gz
EMD-56022,9TKA,2026-07-29,2026-07-29,2.7,le3.0,247,1,0.0,0.7625,9THK,4,0.000,3,293,1,182.3,https://files.rcsb.org/download/9TKA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56022/map/emd_56022.map.gz
EMD-65930,9WFL,2026-04-01,2026-04-01,2.75,le3.0,682,2,0.0,0.8396,9WF3,5,1.000,6,697,2,65.5,https://files.rcsb.org/download/9WFL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65930/map/emd_65930.map.gz
EMD-65077,9VI4,2026-03-11,2026-03-11,1.8,le3.0,516,1,0.0,0.8749,9P4L,6,0.984,9,516,1,1024.0,https://files.rcsb.org/download/9VI4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65077/map/emd_65077.map.gz
EMD-74403,9ZLH,2026-09-16,2026-09-16,2.14,le3.0,206,1,0.0,0.8468,9ZK8,7,1.000,7,206,1,65.5,https://files.rcsb.org/download/9ZLH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74403/map/emd_74403.map.gz
EMD-65363,9VUH,2026-07-15,2026-07-15,2.7,le3.0,634,3,0.0,0.7439,9VUH,8,1.000,1,365,3,128.0,https://files.rcsb.org/download/9VUH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65363/map/emd_65363.map.gz
EMD-70305,9OC0,2026-05-06,2026-05-06,2.97,le3.0,512,2,0.0,0.8262,9OBT,11,1.000,8,862,4,332.8,https://files.rcsb.org/download/9OC0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70305/map/emd_70305.map.gz
EMD-76936,13BD,2026-06-17,2026-06-17,2.65,le3.0,461,1,0.0,0.8841,9OVK,12,0.954,25,1271,2,281.2,https://files.rcsb.org/download/13BD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76936/map/emd_76936.map.gz
EMD-66160,9WQ6,2026-09-16,2026-09-16,3.4,gt3.0_le4.0,261,1,0.0,0.7629,9W3F,4,1.000,11,339,1,128.0,https://files.rcsb.org/download/9WQ6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66160/map/emd_66160.map.gz
EMD-64777,9V4D,2026-07-01,2026-07-01,3.07,gt3.0_le4.0,276,2,0.0,0.8202,25IF,5,0.503,6,393,2,54.0,https://files.rcsb.org/download/9V4D.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64777/map/emd_64777.map.gz
EMD-70828,9OTA,2026-05-27,2026-05-27,3.2,gt3.0_le4.0,534,1,0.0,0.8452,9E94,7,1.000,2,270,2,68.7,https://files.rcsb.org/download/9OTA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70828/map/emd_70828.map.gz
EMD-62507,9KQM,2026-06-03,2026-06-03,3.5,gt3.0_le4.0,403,1,0.0,0.7874,9KKN,8,0.997,9,637,1,65.5,https://files.rcsb.org/download/9KQM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62507/map/emd_62507.map.gz
EMD-66088,9WME,2026-09-23,2026-09-23,3.2,gt3.0_le4.0,446,2,0.0,0.7050,9WME,9,1.000,1,657,2,22.8,https://files.rcsb.org/download/9WME.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66088/map/emd_66088.map.gz
EMD-64920,9VBD,2026-04-08,2026-04-08,3.26,gt3.0_le4.0,521,1,0.0,0.7685,9VBD,15,0.000,2,893,1,128.0,https://files.rcsb.org/download/9VBD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64920/map/emd_64920.map.gz
EMD-54346,9RWS,2026-04-29,2026-04-29,3.28,gt3.0_le4.0,512,1,0.0,0.8467,9RWS,16,0.441,1,551,1,128.0,https://files.rcsb.org/download/9RWS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54346/map/emd_54346.map.gz
EMD-66226,9WTX,2026-09-23,2026-09-23,3.37,gt3.0_le4.0,500,1,0.0,0.7238,9US8,17,0.763,3,551,1,128.0,https://files.rcsb.org/download/9WTX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66226/map/emd_66226.map.gz
EMD-73762,9Z24,2026-06-10,2026-06-10,3.1,gt3.0_le4.0,338,1,0.0,0.8749,9Z1X,18,1.000,4,1214,1,221.2,https://files.rcsb.org/download/9Z24.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73762/map/emd_73762.map.gz
EMD-80558,26DK,2026-08-26,2026-08-26,3.5,gt3.0_le4.0,191,1,0.0,0.7407,26DK,19,0.000,2,197,1,16.0,https://files.rcsb.org/download/26DK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/map/emd_80558.map.gz


In [ ]:
%%writefile scripts/cryozeta_tools.py
#!/usr/bin/env python3
"""CryoZeta helpers of the MICA benchmark (added 2026-11-01 at the request of Prof. Cheng): the parts of the CryoZeta notebook that can be tested without a GPU.

Standard library only. The notebook runs this file with its own Python; CryoZeta itself runs in its own pixi environment (CUDA 11.8 on a T4).

    colabfold_msa()           the MSA of one sequence from the public ColabFold server, written as the two files CryoZeta reads
    emdb_contour_level()      the recommended contour level of a map, from the EMDB header
    write_input_json()        the input file of CryoZeta for one entry
    patch_detection_precision()   the only change of the CryoZeta source that is ever made, and only on a GPU that cannot do bfloat16 (see the function)
    run_entry()               one entry: detection -> CryoZeta -> CryoZeta-Interpolate -> combine, with the fall-backs of the memory ladder

    prepare_input()           the contour level, the MSA of every chain and the input file of one entry (stage 0 of the runner)

    python cryozeta_tools.py run <config.json>        # what the notebook calls for every entry

Why a runner of its own: `cryozeta-inference` catches the error of a sample, writes it to a file and ends with exit code 0, so the exit code says nothing. The runner looks at the files.
The settings of the authors' `inference_demo.sh` (5 samples, 20 diffusion steps, 10 cycles, seed 101) are kept; only the memory-saving knobs may change, in this order:
the attention chunk size (256, 64, 16, 4; it changes the memory and the time, not the algorithm), and the number format (bfloat16, then float32) if a run fails for another reason.
"""
from __future__ import annotations

import argparse
import collections
import glob
import hashlib
import http.client
import io
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

COLABFOLD_HOST = "https://api.colabfold.com"
USER_AGENT = "mica-benchmark-cryozeta/1.0"
AMINO = frozenset("ACDEFGHIKLMNPQRSTVWY")
CHUNK_LADDER = (256, 64, 16, 4)                      # `--infer_setting.chunk_size`; 256 is the default of CryoZeta


class MsaError(RuntimeError):
    pass


# ------------------------------------------------------------------------------------------------------------------------------ MSA
def read_a3m(text):
    """[(header, sequence)] of an a3m text. The NUL characters that the server puts between the results of several queries are removed; wrapped sequences are joined."""
    records, header, parts = [], None, []
    for line in text.replace("\x00", "\n").splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if header is not None:
                records.append((header, "".join(parts)))
            header, parts = line[1:], []
        elif header is not None:
            parts.append(line)
    if header is not None:
        records.append((header, "".join(parts)))
    return records


def write_a3m(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for header, seq in records:
            f.write(">%s\n%s\n" % (header, seq))


def _http(url, data=None, timeout=60.0, tries=3, wait=5.0, sleep=time.sleep):
    body = urllib.parse.urlencode(data).encode("ascii") if data is not None else None
    last = None
    for attempt in range(1, tries + 1):
        try:
            req = urllib.request.Request(url, data=body, headers={"User-Agent": USER_AGENT})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read()
        except (urllib.error.URLError, http.client.HTTPException, OSError, ValueError) as ex:
            last = ex
            if attempt < tries:
                sleep(wait * attempt)
    raise MsaError("%s: %s" % (url, last))


def _json(raw):
    try:
        return json.loads(raw.decode("utf-8", "replace"))
    except ValueError:
        raise MsaError("the MSA server answered with something that is not JSON: %r" % raw[:200])


def _colabfold_once(sequence, host, poll_s, max_wait_s, log, sleep):
    t0 = time.time()
    deadline = t0 + max_wait_s
    while True:                                       # the server may ask us to wait (RATELIMIT) or not know the query yet (UNKNOWN): ask again
        out = _json(_http(host + "/ticket/msa", {"q": ">101\n%s\n" % sequence, "mode": "env"}))
        status = out.get("status")
        if status not in ("UNKNOWN", "RATELIMIT"):
            break
        if time.time() > deadline:
            raise MsaError("the server did not take the query in time (status %s)" % status)
        sleep(poll_s)
    if status in ("ERROR", "MAINTENANCE"):
        raise MsaError("the server answered %s" % status)
    ticket = out.get("id")
    if not ticket:
        raise MsaError("the server gave no ticket: %s" % str(out)[:200])
    while status in ("PENDING", "RUNNING", "UNKNOWN"):
        if time.time() > deadline:
            raise MsaError("the job %s did not finish in %d s (status %s)" % (ticket, max_wait_s, status))
        sleep(poll_s)
        out = _json(_http("%s/ticket/%s" % (host, ticket)))
        status = out.get("status")
    if status != "COMPLETE":
        raise MsaError("the job %s ended with status %s" % (ticket, status))
    blob = _http("%s/result/download/%s" % (host, ticket), timeout=300.0)
    try:
        tf = tarfile.open(fileobj=io.BytesIO(blob), mode="r:*")
    except tarfile.TarError as ex:
        raise MsaError("the result of job %s is not an archive: %s" % (ticket, ex))
    texts = {}
    for member in tf.getmembers():                     # read two members into memory; nothing is extracted to disk
        base = os.path.basename(member.name)
        if member.isfile() and (base == "uniref.a3m" or base.startswith("bfd.mgnify")):
            texts[base] = tf.extractfile(member).read().decode("utf-8", "replace")
    if "uniref.a3m" not in texts:
        raise MsaError("the result of job %s has no uniref.a3m (it has: %s)" % (ticket, ", ".join(sorted(m.name for m in tf.getmembers()))[:300]))
    uni = read_a3m(texts["uniref.a3m"])
    env_name = next((k for k in texts if k.startswith("bfd.mgnify")), None)
    env = read_a3m(texts[env_name]) if env_name else []
    for label, recs in (("uniref.a3m", uni), (env_name, env)):
        if recs and recs[0][1].replace("-", "").upper() != sequence:
            raise MsaError("the first record of %s is not the query" % label)
    return {"ticket": ticket, "uniref": uni, "env": env, "seconds": round(time.time() - t0, 1)}


def colabfold_msa(sequence, out_dir, host=COLABFOLD_HOST, poll_s=8.0, max_wait_s=2400.0, tries=3, log=print, sleep=time.sleep):
    """The MSA of one protein sequence from the ColabFold server (UniRef30 and the environmental databases, the server's `env` mode), as the two files that CryoZeta reads:
    `mmseqs_other_hits.a3m` (the query, the UniRef hits, the environmental hits) and `uniref100_hits.a3m` (the query and the UniRef hits; CryoZeta reads it only to pair the chains
    of a complex of different sequences, and the server's headers carry no species, so no pairs are found; the README of CryoZeta says that it ships no MSA tool and names the
    ColabFold search script). The result is kept in `out_dir`; the same sequence is not asked for twice. Returns the info that is also written to msa_info.json."""
    sequence = "".join(sequence.split()).upper()
    bad = sorted(set(sequence) - AMINO)
    if not sequence or bad:
        raise MsaError("the MSA server takes only the 20 standard amino acids; this sequence has %s" % (bad or "no residues"))
    digest = hashlib.sha256(sequence.encode("ascii")).hexdigest()
    other, uni_path, info_path = (os.path.join(out_dir, n) for n in ("mmseqs_other_hits.a3m", "uniref100_hits.a3m", "msa_info.json"))
    if all(os.path.exists(p) for p in (other, uni_path, info_path)):
        with open(info_path, encoding="utf-8") as f:
            info = json.load(f)
        if info.get("sequence_sha256") == digest:
            log("MSA of %d residues: already here (%d UniRef and %d environmental hits)" % (len(sequence), info["n_uniref_hits"], info["n_env_hits"]))
            return info
    os.makedirs(out_dir, exist_ok=True)
    last = None
    for attempt in range(1, tries + 1):
        try:
            got = _colabfold_once(sequence, host, poll_s, max_wait_s, log, sleep)
            break
        except MsaError as ex:
            last = ex
            log("MSA server, attempt %d of %d: %s" % (attempt, tries, ex))
            if attempt < tries:
                sleep(30.0 * attempt)
    else:
        raise MsaError("the MSA server gave no alignment after %d attempts: %s" % (tries, last))
    uni, env = got["uniref"], got["env"]
    write_a3m(uni, uni_path)
    write_a3m([uni[0]] + uni[1:] + env[1:], other)
    info = {"host": host, "ticket": got["ticket"], "mode": "env", "sequence_length": len(sequence), "sequence_sha256": digest, "n_uniref_hits": len(uni) - 1, "n_env_hits": max(len(env) - 1, 0),
            "server_seconds": got["seconds"], "fetched_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
    log("MSA of %d residues from the ColabFold server: %d UniRef and %d environmental hits (%.0f s)" % (len(sequence), info["n_uniref_hits"], info["n_env_hits"], got["seconds"]))
    return info


# ------------------------------------------------------------------------------------------------------------------------------ the input of CryoZeta
def emdb_header_url(map_url):
    """https://.../EMD-55586/map/emd_55586.map.gz -> https://.../EMD-55586/header/emd-55586.xml"""
    m = re.match(r"^(.*/EMD-(\d+))/map/emd_\2\.map(?:\.gz)?$", map_url)
    if not m:
        raise ValueError("not the address of an EMDB map: %s" % map_url)
    return "%s/header/emd-%s.xml" % (m.group(1), m.group(2))


def emdb_contour_level(xml_text):
    """The recommended contour level of the map from the EMDB header (the primary contour, else the first); None if the header has none."""
    root = ET.fromstring(xml_text)
    found = []
    for el in root.iter():
        if el.tag.split("}")[-1] != "contour":
            continue
        for child in el:
            if child.tag.split("}")[-1] == "level" and child.text and child.text.strip():
                found.append((el.get("primary", "").lower() == "true", float(child.text)))
    if not found:
        return None
    return next((lev for primary, lev in found if primary), found[0][1])


def write_input_json(path, name, map_path, resolution, contour_level, chains):
    """The input of CryoZeta for one entry. chains: [{"sequence": ..., "count": copies, "msa_dir": folder with the two a3m files}], all of them proteins."""
    entry = {"name": name, "modelSeeds": [], "map_path": os.path.abspath(map_path), "resolution": float(resolution), "contour_level": float(contour_level),
             "sequences": [{"proteinChain": {"sequence": c["sequence"], "count": int(c["count"]),
                                             "msa": {"precomputed_msa_dir": os.path.abspath(c["msa_dir"]), "pairing_db": "uniref100"}}} for c in chains]}
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump([entry], f, indent=2)
    return entry


def prepare_input(cfg, say, sleep=time.sleep):
    """Stage 0 of one entry: the recommended contour level (EMDB header, unless cfg has `contour_level`), the MSA of every chain (ColabFold server) and the input file of CryoZeta.
    cfg: name, map_path, resolution, header_url (or contour_level), chains [{"sequence", "count"}], msa_root, input_json, optional colabfold_host. Returns the facts for the summary."""
    info = {"chains": [], "contour_level": None, "contour_level_source": ""}
    level = cfg.get("contour_level")
    if level is None:
        raw = _http(cfg["header_url"], timeout=60.0, sleep=sleep)
        level = emdb_contour_level(raw.decode("utf-8", "replace"))
        if level is None:
            raise ValueError("the EMDB header %s has no recommended contour level" % cfg["header_url"])
        info["contour_level_source"] = cfg["header_url"]
    else:
        info["contour_level_source"] = "given"
    info["contour_level"] = float(level)
    say("contour level of %s: %s (%s)" % (cfg["name"], level, info["contour_level_source"]))
    chains = []
    for i, c in enumerate(cfg["chains"], 1):
        folder = os.path.join(cfg["msa_root"], "chain%d" % i)
        msa = colabfold_msa(c["sequence"], folder, host=cfg.get("colabfold_host") or COLABFOLD_HOST, poll_s=float(cfg.get("msa_poll_s", 8.0)), tries=int(cfg.get("msa_tries", 3)), log=say, sleep=sleep)
        chains.append({"sequence": c["sequence"], "count": c["count"], "msa_dir": folder})
        info["chains"].append({"chain": i, "length": len(c["sequence"]), "copies": int(c["count"]), "n_uniref_hits": msa["n_uniref_hits"], "n_env_hits": msa["n_env_hits"],
                               "ticket": msa.get("ticket", ""), "msa_dir": folder})
    write_input_json(cfg["input_json"], cfg["name"], cfg["map_path"], cfg["resolution"], level, chains)
    return info


# ------------------------------------------------------------------------------------------------------------------------------ the one change of the source
DETECTION_FILE = os.path.join("src", "cryozeta", "runner", "inference_detection.py")
DETECTION_CHANGES = (("self.detection_model.to(self.device).to(torch.bfloat16)", "self.detection_model.to(self.device).to(torch.float32)"),
                     ("amp_dtype = torch.bfloat16", "amp_dtype = torch.float32"))


def patch_detection_precision(repo):
    """The detection network of CryoZeta is converted to bfloat16 and run under bfloat16 autocast without a choice (two lines of `inference_detection.py`). A GPU that cannot do
    bfloat16 (the preflight of the notebook tests a bfloat16 matrix product and 3D convolution) would stop there; on such a GPU the two lines are changed to float32. Nothing else
    is touched, and never on a GPU that can do bfloat16. Every line must occur exactly once (otherwise the source is not the pinned commit). Returns the changed text of the diff."""
    path = os.path.join(repo, DETECTION_FILE)
    with open(path, encoding="utf-8") as f:
        text = f.read()
    new = text
    lines = []
    for old, repl in DETECTION_CHANGES:
        if new.count(repl) == 1 and new.count(old) == 0:
            continue                                                   # changed already
        if new.count(old) != 1:
            raise RuntimeError("%s: the line `%s` occurs %d times (expected once): the source is not the pinned commit" % (DETECTION_FILE, old, new.count(old)))
        new = new.replace(old, repl)
        lines.append("-%s\n+%s" % (old, repl))
    if new != text:
        with open(path, "w", encoding="utf-8") as f:
            f.write(new)
    return "\n".join(lines)


# ------------------------------------------------------------------------------------------------------------------------------ the runner of one entry
class GpuPeak(threading.Thread):
    """Polls nvidia-smi for the memory used on one GPU and keeps the highest value (MB)."""

    def __init__(self, index, every=2.0):
        super().__init__(daemon=True)
        self.index, self.every, self.peak, self.total = index, every, None, None
        self._halt = threading.Event()

    def run(self):
        if not shutil.which("nvidia-smi"):
            return
        while not self._halt.is_set():
            try:
                out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits", "-i", str(self.index)],
                                     capture_output=True, text=True, timeout=20).stdout.strip().splitlines()
                used, total = [int(x) for x in out[0].split(",")]
                self.peak = used if self.peak is None else max(self.peak, used)
                self.total = total
            except Exception:
                pass
            self._halt.wait(self.every)

    def stop(self):
        self._halt.set()
        self.join(timeout=5)


def _physical_gpu():
    first = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0].strip()
    return int(first) if first.isdigit() else 0


def stage_env(cfg):
    """The environment of the CryoZeta programs: what `env_setup.sh` (the activation script of the pixi project) sets, without pixi."""
    env = dict(os.environ)
    for k in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "LAYERNORM_TYPE", "PIXI_ENVIRONMENT_NAME"):
        env.pop(k, None)
    prefix = cfg["env_dir"]
    env["CONDA_PREFIX"] = prefix
    env["PATH"] = os.path.join(prefix, "bin") + os.pathsep + env.get("PATH", "")
    env["LD_LIBRARY_PATH"] = os.path.join(prefix, "lib") + (os.pathsep + env["LD_LIBRARY_PATH"] if env.get("LD_LIBRARY_PATH") else "")
    env["CUTLASS_PATH"] = prefix
    env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
    env["USE_OPM_CHUNKED"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    teaser = cfg.get("teaser_dir")
    if teaser:
        env["TEASERPP_DIR"] = teaser
        env["PYTHONPATH"] = os.path.join(teaser, "build", "python")
    if cfg.get("kernels") == "demo":                                       # the settings of inference_demo.sh (a GPU of the Ampere generation or later, a CUDA compiler)
        env["LAYERNORM_TYPE"] = "fast_layernorm"
        env["CUDA_HOME"] = prefix
    return env


def _flag(on):
    return "true" if on else "false"


def detection_cmd(cfg):
    return [os.path.join(cfg["env_dir"], "bin", "cryozeta-detection"), "json-run", cfg["input_json"], cfg["dump_dir"], "--device", "cuda",
            "--checkpoint-path", os.path.join(cfg["assets"], cfg["weights"]["detection"])]


def inference_cmd(cfg, interpolate, dtype, chunk):
    kernels = cfg.get("kernels") == "demo"
    return [os.path.join(cfg["env_dir"], "bin", "cryozeta-inference"), "--seeds", str(cfg["seed"]),
            "--load_checkpoint_path", os.path.join(cfg["assets"], cfg["weights"]["interpolate" if interpolate else "cryozeta"]),
            "--em_file_dir", cfg["dump_dir"], "--dump_dir", cfg["dump_dir"], "--input_json_path", cfg["input_json"],
            "--use_deepspeed_evo_attention", _flag(kernels), "--use_cuequivariance_attention", _flag(kernels), "--use_cuequivariance_multiplicative_update", _flag(kernels),
            "--use_cuequivariance_attention_pair_bias", _flag(kernels), "--use_opm_tilelang", "false",
            "--model.N_cycle", str(cfg["n_cycle"]), "--sample_diffusion.N_sample", str(cfg["n_sample"]), "--sample_diffusion.N_step", str(cfg["n_step"]),
            "--data.num_dl_workers", "1", "--use_interpolation", _flag(interpolate), "--overwrite", "true",
            "--dtype", dtype, "--infer_setting.chunk_size", str(chunk)]


def combine_cmd(cfg):
    return [os.path.join(cfg["env_dir"], "bin", "cryozeta-combine"), "--dump-dir", cfg["dump_dir"], "--input-json-path", cfg["input_json"], "--seeds", str(cfg["seed"]),
            "--num-select", str(cfg["num_select"])]


def run_stage(cfg, label, cmd, env, say):
    """Runs one program, streams its output (and writes it to logs/<label>.log), polls the GPU memory. Returns what happened; `oom` is true if the output says so anywhere."""
    log_dir = os.path.join(cfg["dump_dir"], "logs")
    os.makedirs(log_dir, exist_ok=True)
    gpu = GpuPeak(_physical_gpu())
    gpu.start()
    t0 = time.time()
    tail, oom, bars, last_progress = collections.deque(maxlen=60), False, 0, 0.0
    say("\n=== stage %s: %s" % (label, " ".join(os.path.basename(cmd[0]) if i == 0 else a for i, a in enumerate(cmd))[:500]))
    try:
        with open(os.path.join(log_dir, "%s.log" % label), "a", encoding="utf-8") as lf:
            lf.write("$ %s\n" % " ".join(cmd))
            try:
                p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, cwd=cfg["repo"], errors="replace")
            except OSError as ex:
                msg = "could not start %s: %s" % (cmd[0], ex)
                lf.write(msg + "\n")
                say(msg)
                return {"stage": label, "rc": 127, "seconds": 0.0, "gpu_peak_mb": None, "oom": False, "tail": [msg]}
            for line in p.stdout:
                line = line.rstrip("\n")
                lf.write(line + "\n")
                lf.flush()
                tail.append(line[:400])
                if "out of memory" in line.lower():
                    oom = True
                if "%|" in line:
                    bars += 1
                    if bars % 50 != 1:
                        continue
                say(line[:400])
                if time.time() - last_progress > 1.0:
                    last_progress = time.time()
                    _progress(cfg, "%s: %d s | %s" % (label, time.time() - t0, line.strip()[:160]))
            p.wait()
            p.stdout.close()
    finally:
        gpu.stop()
    return {"stage": label, "rc": p.returncode, "seconds": round(time.time() - t0, 1), "gpu_peak_mb": gpu.peak, "gpu_total_mb": gpu.total, "oom": oom, "tail": list(tail)}


def _progress(cfg, text):
    try:
        with open(cfg["progress_path"], "w", encoding="utf-8") as f:
            f.write(text + "\n")
    except OSError:
        pass


def _entry_dir(cfg, *parts):
    return os.path.join(cfg["dump_dir"], cfg["name"], *parts)


def _predictions(cfg, stage_dir):
    return sorted(glob.glob(os.path.join(_entry_dir(cfg, stage_dir, "seed_%s" % cfg["seed"], "predictions"), "*_sample_*.cif")))


def _error_text(cfg, stage_dir):
    p = _entry_dir(cfg, stage_dir, "ERR", "%s.txt" % cfg["name"])
    if os.path.exists(p):
        with open(p, encoding="utf-8", errors="replace") as f:
            return f.read()
    return ""


def _inference(cfg, interpolate, state, env, say, attempts):
    """CryoZeta (or CryoZeta-Interpolate) with the memory ladder; returns True when the predictions exist. `state` carries the settings that worked to the next stage."""
    label = "interpolate" if interpolate else "cryozeta"
    stage_dir = "CryoZeta-Interpolate" if interpolate else "CryoZeta"
    ladder = list(cfg.get("chunk_sizes") or CHUNK_LADDER)
    while len(attempts) < 8:
        chunk = ladder[state["chunk_index"]]
        shutil.rmtree(_entry_dir(cfg, stage_dir), ignore_errors=True)       # the files of an earlier attempt (its error file above all) must not be taken for the result of this one
        res = run_stage(cfg, "%s_try%d" % (label, len([a for a in attempts if a["stage"] == label]) + 1), inference_cmd(cfg, interpolate, state["dtype"], chunk), env, say)
        files = _predictions(cfg, stage_dir)
        err = _error_text(cfg, stage_dir)
        oom = res["oom"] or "out of memory" in err.lower()
        skipped = interpolate and not files and not err and any("no interpolation features" in ln for ln in res["tail"])
        ok = bool(files) and res["rc"] == 0 and not err
        attempts.append({"stage": label, "dtype": state["dtype"], "chunk_size": chunk, "rc": res["rc"], "seconds": res["seconds"], "gpu_peak_mb": res["gpu_peak_mb"], "oom": oom,
                         "predictions": len(files), "error": err[-600:] if err else "", "skipped": skipped})
        if ok or skipped:
            return True
        res["error_text"] = err
        attempts[-1]["tail"] = res["tail"][-12:]
        if oom and state["chunk_index"] + 1 < len(ladder):
            state["chunk_index"] += 1
            say("%s: out of GPU memory; the next attempt uses attention chunk size %d" % (label, ladder[state["chunk_index"]]))
            continue
        if not oom and state["dtype"] != "fp32":
            state["dtype"] = "fp32"
            say("%s: failed (not for memory); the next attempt computes in float32" % label)
            continue
        return False
    return False


def _copy_small_files(cfg):
    dst = os.path.join(cfg["dump_dir"], "summary_files")
    os.makedirs(dst, exist_ok=True)
    pairs = [(_entry_dir(cfg, "CryoZeta-Detection", "%s_timing.txt" % cfg["name"]), "detection_timing.txt"),
             (_entry_dir(cfg, "CryoZeta", "saved_data", "scores.csv"), "cryozeta_scores.csv"),
             (_entry_dir(cfg, "CryoZeta-Interpolate", "saved_data", "scores.csv"), "interpolate_scores.csv")]
    for stage_dir, tag in (("CryoZeta", "cryozeta"), ("CryoZeta-Interpolate", "interpolate")):
        for p in sorted(glob.glob(os.path.join(_entry_dir(cfg, stage_dir, "seed_%s" % cfg["seed"], "predictions"), "*summary_confidence*.json")))[:10]:
            pairs.append((p, "%s_%s" % (tag, os.path.basename(p))))
    for src, name in pairs:
        if os.path.exists(src) and os.path.getsize(src) < 2000000:
            shutil.copyfile(src, os.path.join(dst, name))


def run_entry(cfg, say=None):
    """One entry from the detection to the final models. Writes driver_summary.json; returns (exit code, summary): 0 the final models exist, 2 failed for lack of GPU memory, 3 failed."""
    say = say or (lambda s: print(s, flush=True))
    t0 = time.time()
    os.makedirs(cfg["dump_dir"], exist_ok=True)
    env = stage_env(cfg)
    state = {"dtype": cfg["dtype"], "chunk_index": 0}
    stages, attempts = [], []
    summary = {"name": cfg["name"], "status": "failed", "failed_stage": "", "note": "", "settings": {k: cfg.get(k) for k in ("seed", "n_sample", "n_step", "n_cycle", "num_select", "dtype", "kernels", "chunk_sizes")},
               "stages": stages, "attempts": attempts}
    failing = None
    _progress(cfg, "input: contour level and MSA")
    try:
        summary["input"] = prepare_input(cfg, say)
    except Exception as ex:                                                  # the server, the header or the sequence: a failure of the input, not of the method
        failing = ("input", {"tail": [str(ex)[:400]], "oom": False})
        summary["note"] = "input: %s: %s" % (type(ex).__name__, str(ex)[:400])
    if failing is None:
        _progress(cfg, "detection: starting")
        det = run_stage(cfg, "detection", detection_cmd(cfg), env, say)
        stages.append({k: v for k, v in det.items() if k != "tail"})
        if det["rc"] != 0 or not os.path.exists(_entry_dir(cfg, "CryoZeta-Detection", "%s.pt" % cfg["name"])):
            failing = ("detection", det)
            summary["note"] = "detection: exit code %s%s%s" % (det["rc"], "; no detection file" if det["rc"] == 0 else "", "; CUDA out of memory" if det["oom"] else "")
    if failing is None:
        for interpolate in (False, True):
            label = "interpolate" if interpolate else "cryozeta"
            _progress(cfg, "%s: starting" % label)
            ok = _inference(cfg, interpolate, state, env, say, attempts)
            if not ok:
                last = attempts[-1]
                failing = (label, {"tail": last.get("tail", []), "oom": last["oom"]})
                summary["note"] = "%s: no predictions after %d attempt(s) (%s)" % (label, len([a for a in attempts if a["stage"] == label]), "CUDA out of memory" if last["oom"] else (last["error"][-200:] or "exit code %s" % last["rc"]))
                break
    if failing is None:
        _progress(cfg, "combine: starting")
        comb = run_stage(cfg, "combine", combine_cmd(cfg), env, say)
        stages.append({k: v for k, v in comb.items() if k != "tail"})
        final = _entry_dir(cfg, "CryoZeta-Final", "%s_sample_0.cif" % cfg["name"])
        if comb["rc"] != 0 or not os.path.exists(final):
            failing = ("combine", comb)
            summary["note"] = "combine: exit code %s%s" % (comb["rc"], "; no final model" if comb["rc"] == 0 else "")
    _copy_small_files(cfg)
    summary["seconds"] = round(time.time() - t0, 1)
    summary["final_settings"] = dict(state, chunk_size=(cfg.get("chunk_sizes") or list(CHUNK_LADDER))[state["chunk_index"]])
    summary["gpu_peak_mb"] = max([s.get("gpu_peak_mb") or 0 for s in stages + attempts] or [0]) or None
    if failing is None:
        summary["status"] = "ok"
        code = 0
        say("\nDRIVER RESULT ok: %s, %.0f s, GPU memory peak %s MB, settings %s" % (cfg["name"], summary["seconds"], summary["gpu_peak_mb"], summary["final_settings"]))
    else:
        summary["failed_stage"] = failing[0]
        oom = bool(failing[1].get("oom"))
        code = 2 if oom else 3
        say("\nDRIVER RESULT failed: %s. The last lines of the stage:" % summary["note"])
        for line in (failing[1].get("tail") or [])[-25:]:
            say("    " + line)
        say("DRIVER RESULT failed: %s" % summary["note"])
    with open(cfg["summary_path"], "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    _progress(cfg, "done: %s" % summary["status"])
    return code, summary


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    r = sub.add_parser("run", help="one entry, from a JSON file of settings")
    r.add_argument("config")
    args = ap.parse_args(argv)
    with open(args.config, encoding="utf-8") as f:
        cfg = json.load(f)
    code, _summary = run_entry(cfg)
    return code


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
EXPECTED = {'scripts/nbtools.py': '9bda95fce7f7fbe0c373e315740f555fc27942a08971ce2fe1b5cae6af6d748d', 'scripts/mica_inputs.py': '87b7f5e43b86671e09e673e8bb07abb20496ad92264528dfd9994b88bded417f', 'scripts/evaluate_models.py': '8c235732d2595f8ad2014d70a278b44097205aff2417c9b5761f55dfe4974250', 'data/af3/af3_jobs.csv': 'b43068ffed7e3843d0d9c5cfa509d8f1ba9900527a1a0cad9e74aa69573cd4b0', 'data/af3/af3_files_sha256.txt': 'c131836802f9469c2073cc8c0bbb495e906854ffb223100cf30750714e2e3a4f', 'data/entries_2026.csv': 'cb695bb04ef09bf48f8c2848f91a2580c0b3c15f2a8f1e580659d86f0097a5da', 'scripts/cryozeta_tools.py': '7d74427f8e52a991e2b1f0eb8f9f454fd7ac296899b4c77d732c6c716d2b25e8'}
bad = [p for p, h in EXPECTED.items() if not os.path.exists(p) or sha256_file(p) != h]
if bad:
    fail("these files were not written as expected: %s. Use Run all again; if it happens again, send me this message." % ", ".join(bad))
else:
    log("all %d files written and checked (hashes match)" % len(EXPECTED))


## Step 2b. Progress lines, a zip that is refreshed, and the last cell that works after a restart

This cell adds three things that the first pilot lacked: a progress line every few minutes while a program is silent, the file `state.json`, and a zip
(`gpu_cryozeta_a_partial.zip` in the output files) that is written again after every stage and every ten minutes, so a stop in the middle never leaves nothing.

In [ ]:
sys.path.insert(0, os.path.join(WORK, "scripts"))
import nbtools

# Kaggle's kernel sets MPLBACKEND=module://matplotlib_inline.backend_inline. Every program started from this notebook inherits it, and `import matplotlib.pyplot`
# then fails in any environment without matplotlib-inline (ModelAngelo's, in the first pilot: the run stopped after 5 minutes). Agg needs nothing.
os.environ["MPLBACKEND"] = "Agg"

NOTEBOOK = "gpu_cryozeta_a"
STATE_FILE = os.path.join(WORK, "state.json")
PARTIAL_ZIP = nbtools.partial_zip_path(OUT_DIR, NOTEBOOK)
PACK_LOCK = threading.Lock()
LAST_PARTIAL = {"t": 0.0}


def save_state():
    try:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(STATE, f, indent=1, default=str)
    except Exception as ex:
        log("could not save state.json: %s" % ex)


def pack_now(min_gap=0):
    """Writes the partial zip again (same name every time). min_gap: do nothing if the last one is younger than this many seconds."""
    with PACK_LOCK:
        if time.time() - LAST_PARTIAL["t"] < min_gap:
            return
        LAST_PARTIAL["t"] = time.time()
        try:
            save_state()
            nbtools.pack(WORK, PARTIAL_ZIP, NOTEBOOK + "_partial", secrets=SECRETS)
        except Exception as ex:
            log("the partial zip could not be written: %s" % ex)


BEAT = float(os.environ.get("MICA_BEAT", "120"))          # seconds between progress lines (the tests make it short)


def run_watched(cmd, label, watch=(), beat=None, zip_every=600, **kw):
    """run() with a progress line every `beat` seconds (time, load, free RAM, busiest processes, last line of the files in `watch`)."""
    hb = nbtools.Heartbeat(label, log, watch=watch, every=beat or BEAT, on_tick=lambda: pack_now(zip_every))
    hb.start()
    try:
        return run(cmd, **kw)
    finally:
        hb.stop()


def finish(complete, extra_info=None):
    """Writes run_info.json and the final zip; returns the path of the zip."""
    info = {"notebook": NOTEBOOK, "started_utc": STATE["started"], "finished_utc": nbtools.utc_now().isoformat(), "elapsed_minutes": round((time.time() - STATE["t0"]) / 60),
            "problem": STATE["problem"], "complete": complete, "platform": PLATFORM, "python": platform.python_version(), "os": platform.platform(), "gpu": STATE["gpu"],
            "cpu_cores": os.cpu_count(), "ram_gb": meminfo_gb(), "state": STATE}
    info.update(extra_info or {})
    with open(os.path.join(WORK, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    save_state()
    path, n, left_out = nbtools.final_pack(WORK, OUT_DIR, NOTEBOOK, complete, secrets=SECRETS)
    log("zip written: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        log("left out of the zip: %s" % "; ".join(left_out))
    return path


def cpu_facts():
    """How many processors the program sees and how many it may really use (a container can show more cores than its limit allows)."""
    import multiprocessing
    facts = {"os.cpu_count": os.cpu_count(), "multiprocessing.cpu_count": multiprocessing.cpu_count(),
             "usable (sched_getaffinity)": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None, "nproc": sh("nproc 2>&1"),
             "cgroup v2 cpu.max": sh("cat /sys/fs/cgroup/cpu.max 2>&1"), "cgroup v1 quota": sh("cat /sys/fs/cgroup/cpu/cpu.cfs_quota_us 2>&1"),
             "MICA's nproc formula min(max(int(0.6 * cpu_count), 1), 48)": min(max(int(0.6 * multiprocessing.cpu_count()), 1), 48)}
    with open(os.path.join(WORK, "cpu_facts.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join("%s: %s" % kv for kv in facts.items()) + "\n")
    log("processors: " + " | ".join("%s %s" % (k.split(" ")[0], v) for k, v in facts.items()))
    STATE["cpu_facts"] = facts


save_state()
cpu_facts()
log("progress lines, state.json and the partial zip (%s) are ready" % PARTIAL_ZIP)


## Step 3. Tools shared by the methods, the schedule and the limits

`uv` and the environment builder, the downloader, the archive helpers, the program runner (time limit, GPU and RAM log of the GPU it runs on, a progress line every two
minutes), the scoring, and the scheduler: a list of tasks, one worker per GPU, the rules that keep the two memory-hungry methods apart and that stop the work in time.
Nothing is installed or downloaded here yet.

In [ ]:
sys.path.insert(0, os.path.join(WORK, "scripts"))
import evaluate_models, mica_inputs

UV_VERSION = "0.13.1"
ENV_PY_GIVEN = None                       # None: build the environments; a path is only used by the tests
IMPORT_TESTS = None                     # None: the import test of each method (a dict is only used by the tests)
SPECS = {'cryozeta': {'key': 'cryozeta', 'label': 'CryoZeta', 'repo': 'https://github.com/kiharalab/CryoZeta', 'commit': 'c544c6bbd0366951d8b4e0bc4c0da547836f7b8f', 'python': '3.12', 'req': None, 'timeout': 21600, 's_per_residue': 15, 'oom_is_result': True, 'uses_af3': False, 'bundle': None, 'weights_url': None, 'weights_min_bytes': 0, 'gdrive_id': None, 'import_test': "import importlib.util, torch, safetensors; assert importlib.util.find_spec('cryozeta') is not None; print('the CryoZeta environment imports: torch ' + torch.__version__)", 'pixi': {'version': '0.41.1', 'url': 'https://github.com/prefix-dev/pixi/releases/download/v0.41.1/pixi-x86_64-unknown-linux-musl', 'min_bytes': 30000000, 'sha256': 'f4e98919053c099c6839602c3334e1b7802d4db445a2ffa9282ff6abac6c0418', 'env': 'cu11'}, 'teaser': {'repo': 'https://github.com/MIT-SPARK/TEASER-plusplus', 'commit': '52a9c52ee7d4c838c5e8a75458c33178be5bfb70', 'cmake': '3.31.6', 'pybind11': '3.0.1', 'prebuilt': None}, 'hub': {'repo': 'KiharaLab/CryoZeta', 'api': 'https://huggingface.co/api/models/KiharaLab/CryoZeta?blobs=true', 'resolve': 'https://huggingface.co/KiharaLab/CryoZeta/resolve/{rev}/{path}', 'min_bytes': 1000000, 'files': {'detection': 'cryozeta-detection-v0.0.1.safetensors', 'cryozeta': 'cryozeta-v0.0.1.safetensors', 'interpolate': 'cryozeta-interpolate-v0.0.1.safetensors'}}, 'colabfold': 'https://api.colabfold.com', 'msa_poll_s': 8.0, 'msa_tries': 3, 'run': {'seed': 101, 'n_sample': 5, 'n_step': 20, 'n_cycle': 10, 'num_select': 5, 'dtype': 'bf16', 'kernels': 'off'}}}
RUN_METHODS = ['cryozeta']                       # the order in which the methods run; delete a name to leave that method out
ENTRIES = ['9T5K', '9WQ6', '22FX', '9V4D', '9JKK', '9OTA', '9TKA', '9KQM', '9WFL', '9WME', '9VI4', '9VBD', '9ZLH', '9RWS', '9VUH', '9WTX', '9OC0', '9Z24', '13BD', '26DK']
USALIGN_REPO = "https://github.com/pylelab/USalign"
USALIGN_COMMIT = "746ad8a2a4dea9f317075cc482df0ecb5440bdea"
USALIGN_GIVEN = None                     # None: compile it; a path is only used by the tests
TORCH_HOME = os.path.join(BIG, "torch_home")      # the ESM weights and other downloads of torch.hub go here
RETRY_WAIT = float(os.environ.get("MICA_RETRY_WAIT", "10"))          # seconds; the tests make it short
PRIVATE_KINDS = ("emprot_fit",)                  # models that are the AlphaFold Server predictions placed into the map: scored, but not copied into the zip
INPUT_ROOT = os.path.join(BIG, "input")
RUN_ROOT = os.path.join(BIG, "runs")
os.makedirs(TORCH_HOME, exist_ok=True)
os.makedirs(os.path.join(WORK, "logs"), exist_ok=True)
STATE.setdefault("methods", {})
UV = {}
GPU_TEST = r"""
import torch
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(3)
p = torch.cuda.get_device_properties(0)
print("gpu: %s | %.1f GB | capability %d.%d" % (p.name, p.total_memory / 2**30, p.major, p.minor))
x = torch.randn(1, 4, 24, 24, 24, device="cuda")
y = torch.nn.Conv3d(4, 8, 3, padding=1).cuda()(x)
a = torch.randn(512, 512, device="cuda")
z = float((a @ a).sum()) + float(y.sum())
torch.cuda.synchronize()
print("3D convolution and matrix product on the GPU work (torch %s, cuda %s)" % (torch.__version__, torch.version.cuda))
"""


def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path, columns, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns, lineterminator="\n", extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def download(url, dest, min_bytes=1, tries=3):
    import requests
    if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    last = ""
    for attempt in range(1, tries + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 120)) as r:
                r.raise_for_status()
                done = 0
                with open(dest + ".part", "wb") as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
                        done += len(block)
            if done < min_bytes:
                raise IOError("only %d bytes arrived" % done)
            os.replace(dest + ".part", dest)
            return dest
        except (requests.RequestException, IOError) as ex:
            last = str(ex)[:200]
            log("download attempt %d of %d failed (%s): %s" % (attempt, tries, url, last))
            time.sleep(5 * attempt)
    raise RuntimeError("could not download %s: %s" % (url, last))


def git_checkout(repo, commit, folder):
    if os.path.isdir(os.path.join(folder, ".git")) and sh("git -C %s rev-parse HEAD" % folder) == commit:
        return True
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)
    for cmd in (["git", "init", "-q", folder], ["git", "-C", folder, "remote", "add", "origin", repo],
                ["git", "-C", folder, "fetch", "-q", "--depth", "1", "origin", commit], ["git", "-C", folder, "checkout", "-q", "FETCH_HEAD"]):
        rc, last, _ = run(cmd)
        if rc != 0:
            log("\n".join(last))
            return False
    return sh("git -C %s rev-parse HEAD" % folder) == commit


def build_usalign():
    if USALIGN_GIVEN:
        STATE["usalign"] = USALIGN_GIVEN
    else:
        src = os.path.join(BIG, "USalign_src")
        exe = os.path.join(src, "USalign")
        if not os.path.exists(exe):
            if not git_checkout(USALIGN_REPO, USALIGN_COMMIT, src):
                fail("US-align could not be fetched at commit %s" % USALIGN_COMMIT)
                return False
            rc, last, _ = run(["g++", "-O3", "-ffast-math", "-o", exe, os.path.join(src, "USalign.cpp"), "-lm"], cwd=src, every=10)
            if rc != 0:
                fail("US-align could not be compiled (see the lines above). Send me the zip.")
                return False
        STATE["usalign"] = exe
    r = subprocess.run([STATE["usalign"], "-h"], capture_output=True, text=True, errors="replace")
    head = [ln for ln in (r.stdout + r.stderr).splitlines() if "Version" in ln]
    STATE["usalign_version"] = head[0].strip() if head else "unknown"
    log("US-align: %s (%s)" % (STATE["usalign"], STATE["usalign_version"]))
    return True


def find_af3_dir(rows):
    """The folder that holds the AlphaFold Server files of this entry. Looks in the Kaggle inputs, unpacks a zip if needed, and on Colab asks for the zip."""
    bases = [b for b in (os.environ.get("MICA_AF3_DIR"), "/kaggle/input", os.path.join(WORK, "af3_upload"), WORK) if b and os.path.isdir(b)]

    def complete(root):
        try:
            for r in rows:
                mica_inputs.find_model(root, r["job_name"])
            return True
        except ValueError:
            return False

    def search():
        for b in bases:
            if complete(b):
                return b
        for b in bases:
            for z in sorted(glob.glob(os.path.join(b, "**", "*.zip"), recursive=True)):
                name = os.path.basename(z).lower()
                if os.path.dirname(os.path.abspath(z)) == os.path.abspath(OUT_DIR) or not ("alpha" in name or "af3" in name or "fold" in name):
                    continue
                target = os.path.join(WORK, "af3_unzipped", os.path.splitext(os.path.basename(z))[0])
                if not os.path.isdir(target):
                    with zipfile.ZipFile(z) as zf:
                        zf.extractall(target)
                if complete(target):
                    return target
        return None

    found = search()
    if not found and IN_COLAB:
        log("The AlphaFold Server files were not found. A file chooser appears now: choose AlphaFold_ServerJOBS.zip (the zip with the 26 fold_*_model_0.cif files).")
        try:
            os.makedirs(os.path.join(WORK, "af3_upload"), exist_ok=True)
            os.chdir(os.path.join(WORK, "af3_upload"))
            from google.colab import files
            files.upload()
        finally:
            os.chdir(WORK)
        bases.append(os.path.join(WORK, "af3_upload"))
        found = search()
    if not found:
        raise RuntimeError("the AlphaFold Server files of this entry (%s) were not found. Kaggle: Add Input -> Upload -> New Dataset, upload "
                           "AlphaFold_ServerJOBS.zip, keep it private, and Run all again. Colab: run the cell again and choose the zip when the file chooser appears."
                           % ", ".join("fold_%s_model_0.cif" % r["job_name"].lower() for r in rows))
    return found


def ensure_uv():
    if UV:
        return True
    rc, last, _ = run([sys.executable, "-m", "pip", "install", "-q", "uv==" + UV_VERSION])
    if rc != 0:
        return False
    UV.update(cmd=[sys.executable, "-m", "uv"],
              env={"UV_CACHE_DIR": os.environ.get("UV_CACHE_DIR") or os.path.join(BIG, "uv_cache"),
                   "UV_PYTHON_INSTALL_DIR": os.environ.get("UV_PYTHON_INSTALL_DIR") or os.path.join(BIG, "uv_python"), "UV_PYTHON_PREFERENCE": "only-managed"})
    log("uv: " + sh("%s -m uv --version" % sys.executable))
    return True


def env_vars(env_py, extra=None):
    """The environment variables for a program of a method: its own python first on the PATH, torch.hub folder, the first GPU only."""
    e = {"PATH": os.path.dirname(env_py) + os.pathsep + os.environ.get("PATH", ""), "VIRTUAL_ENV": None, "PYTHONPATH": None, "PYTHONHOME": None,
         "TORCH_HOME": TORCH_HOME, "CUDA_VISIBLE_DEVICES": "0", "MPLBACKEND": "Agg"}
    e.update(extra or {})
    return e


def pinned(req_file):
    with open(req_file, encoding="utf-8") as f:
        return [ln.strip() for ln in f if "==" in ln and not ln.startswith("#")]


def make_env(spec, repo):
    """Builds the Python environment of a method from its pins (uv, PyPI wheels); returns the path of its python, or raises."""
    key, test = spec["key"], (IMPORT_TESTS or {}).get(spec["key"]) or spec["import_test"]
    if ENV_PY_GIVEN:
        return ENV_PY_GIVEN
    env_dir = os.path.join(BIG, "env_" + key)
    env_py = os.path.join(env_dir, "bin", "python")
    if os.path.exists(env_py):
        rc, last, _ = run([env_py, "-c", test], env=env_vars(env_py), cwd=repo, quiet=True)
        if rc == 0:
            log("the environment of %s exists already: %s" % (spec["label"], last[-1]))
            return env_py
        shutil.rmtree(env_dir, ignore_errors=True)
    if not ensure_uv():
        raise RuntimeError("pip could not install uv (see the lines above)")
    uv = UV["cmd"]
    for what, cmd in (("Python " + spec["python"], uv + ["python", "install", spec["python"]]),
                      ("the virtual environment", uv + ["venv", "--python", spec["python"], env_dir]),
                      ("the %d packages of %s (a few minutes)" % (len(pinned(spec["req"])), spec["req"]), uv + ["pip", "install", "--python", env_py, "--no-deps", "-r", spec["req"]])):
        log("installing " + what + " ...")
        rc, last, _ = run_watched(cmd, "uv: " + what[:30], env=UV["env"], every=20)
        if rc != 0:
            raise RuntimeError("uv could not install %s (exit code %d, the last lines are above)" % (what, rc))
    rc, last, _ = run([env_py, "-c", test], env=env_vars(env_py), cwd=repo)
    if rc != 0 and any("libGL" in ln or "libgomp" in ln for ln in last):
        log("a system library is missing; installing it with apt ...")
        run(["bash", "-c", "apt-get update -qq && apt-get install -y -qq libgl1 libgomp1"], every=20)
        rc, last, _ = run([env_py, "-c", test], env=env_vars(env_py), cwd=repo)
    if rc != 0:
        raise RuntimeError("the packages of %s were installed but the method's own modules do not import (see the lines above)" % spec["label"])
    log(last[-1])
    return env_py


def gpu_test(spec, env_py):
    rc, last, _ = run([env_py, "-c", GPU_TEST], env=env_vars(env_py))
    if rc == 3:
        raise RuntimeError("PyTorch in the environment of %s does not see the GPU" % spec["label"])
    if rc != 0:
        raise RuntimeError("the GPU of this session cannot run the PyTorch of %s (see the lines above)" % spec["label"])


def fetch(url, dest, min_bytes, label, insecure=False, expect_size=None, max_seconds=5400):
    """Downloads with curl (resumes a partial file, 3 attempts, gives up on a connection slower than 20 kB/s for 5 minutes); returns True when the file has at least min_bytes
    (and expect_size bytes if given)."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)

    def size():
        return os.path.getsize(dest) if os.path.exists(dest) else 0

    def good():
        return size() >= min_bytes and (expect_size is None or size() == expect_size)

    if good():
        log("%s: already downloaded (%d MB)" % (label, size() // 1000000))
        return True
    cmd = ["curl", "-sS", "-L", "--fail", "--connect-timeout", "60", "--speed-limit", "20000", "--speed-time", "300", "--max-time", str(max_seconds), "-C", "-", "-o", dest, url]
    if insecure:
        cmd.insert(1, "-k")                                  # the host of the CryoAtom weights has a certificate that wget of the authors also ignores (--no-check-certificate)
    log("downloading %s from %s" % (label, url))
    hb = nbtools.Heartbeat(label, log, every=BEAT, on_tick=lambda: log("   %s: %d MB so far" % (label, size() // 1000000)))
    hb.start()
    t0 = time.time()
    try:
        for attempt in (1, 2, 3):
            rc, last, timed_out = run(cmd, timeout=max_seconds + 300, quiet=True)
            if good():
                break
            if rc == 0 and os.path.exists(dest):
                os.remove(dest)                                  # a complete download of the wrong size (a web page instead of the file): do not resume it
            if rc == 0:
                log("download attempt %d of 3: the server sent a file of the wrong size" % attempt)
                time.sleep(RETRY_WAIT * attempt)
                continue
            log("download attempt %d of 3 failed: curl exit code %s %s" % (attempt, rc, " ".join(last[-2:])[:300]))
            time.sleep(RETRY_WAIT * attempt)
    finally:
        hb.stop()
    log("%s: %d MB in %d s" % (label, size() // 1000000, time.time() - t0))
    return good()


def fetch_gdrive(file_id, dest, min_bytes, label):
    """Downloads a file of Google Drive with gdown (pip installs it into this notebook's own Python)."""
    if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
        return True
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    run([sys.executable, "-m", "pip", "install", "-q", "gdown"], quiet=True)
    for attempt in (1, 2):
        rc, last, timed_out = run_watched([sys.executable, "-m", "gdown", "https://drive.google.com/uc?id=" + file_id, "-O", dest], label, timeout=3 * 3600, every=1)
        if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
            return True
        log("Google Drive download attempt %d of 2 failed (exit code %s): %s" % (attempt, rc, " ".join(last[-3:])[:400]))
        time.sleep(3 * RETRY_WAIT)
    return False


def unpack(archive, dest):
    os.makedirs(dest, exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as t:
            try:
                t.extractall(dest, filter="data")
            except TypeError:                                    # a Python without the filter argument
                t.extractall(dest)
            except tarfile.FilterError:                          # a link or a path that the safe filter refuses; the archive is the authors' own weights file
                t.extractall(dest, filter="tar")
    else:
        with open(archive, "rb") as f:
            head = f.read(200)
        raise RuntimeError("%s is not a zip or tar archive (it starts with %r); Google Drive or the server may have sent a web page instead of the file" % (os.path.basename(archive), head[:80]))


def find_files(root, names):
    """{name: path} of the first file called `name` under root, for each name that exists."""
    found = {}
    for dirpath, _dirs, files in os.walk(root):
        for n in files:
            if n in names and n not in found:
                found[n] = os.path.join(dirpath, n)
    return found


def input_file(name):
    """A file that the user attached as a Kaggle dataset (a way to give a weights file that cannot be downloaded)."""
    for base in [b for b in (os.environ.get("MICA_EXTRA_INPUT"), "/kaggle/input") if b and os.path.isdir(b)]:
        hit = find_files(base, {name}).get(name)
        if hit:
            return hit
    return None


def place_files(archive, names, dest_dir, key):
    """Unpacks `archive` and copies the files `names` (found anywhere in it) into dest_dir."""
    tmp = os.path.join(BIG, "unpack_" + key)
    shutil.rmtree(tmp, ignore_errors=True)
    unpack(archive, tmp)
    found = find_files(tmp, set(names))
    missing = [n for n in names if n not in found]
    if missing:
        listing = sorted(os.path.relpath(os.path.join(d, f), tmp) for d, _s, fs in os.walk(tmp) for f in fs)
        raise RuntimeError("the archive %s lacks %s; it contains %s" % (os.path.basename(archive), ", ".join(missing), listing[:12]))
    os.makedirs(dest_dir, exist_ok=True)
    for n in names:
        shutil.copyfile(found[n], os.path.join(dest_dir, n))
    shutil.rmtree(tmp, ignore_errors=True)
    try:
        os.remove(archive)                                       # the weights are in place; the archive would only use disk space
    except OSError:
        pass


def record_files(key, folders, label="weights"):
    """Logs and remembers the big files (1 MB or more) under `folders`; returns their total size in MB."""
    files = {}
    for folder in folders:
        for dirpath, _dirs, names in os.walk(folder):
            for n in names:
                p = os.path.join(dirpath, n)
                if os.path.getsize(p) >= 1000000:
                    files[os.path.relpath(p, BIG)] = os.path.getsize(p)
    STATE["methods"][key][label] = files
    mb = sum(files.values()) // 1000000
    log("%s files of %s: %d files, %d MB %s" % (label, key, len(files), mb, {k: v // 1000000 for k, v in list(files.items())[:8]}))
    return mb


def chmod_programs(folder, exclude_suffix=(".py", ".md", ".txt", ".json", ".cif", ".pdb", ".so", ".h", ".c", ".cpp", ".pth", ".pt", ".ckpt")):
    """Makes the programs that a repository ships (git may have kept them without the executable bit) executable."""
    n = 0
    for dirpath, _dirs, names in os.walk(folder):
        for f in names:
            p = os.path.join(dirpath, f)
            if not f.lower().endswith(exclude_suffix) and os.path.isfile(p) and not os.access(p, os.X_OK):
                with open(p, "rb") as fh:
                    if fh.read(4) == b"\x7fELF":
                        os.chmod(p, 0o755)
                        n += 1
    return n



def run_program(label, cmd, cwd, env, logname, gpu, watch=(), timeout=7200):
    """Runs a method's program with a time limit, a progress line every two minutes and a log of the memory of its GPU and of the free RAM;
    returns (exit code, last lines, timed_out, info). The output of the program goes to its log file (in the zip), not to the notebook."""
    logf = os.path.join(WORK, "logs", logname)
    mon = GpuMonitor(os.path.join(WORK, "gpu", logname.replace(".log", ".csv")), gpu)
    mon.start()
    t0 = time.time()
    rc, last, timed_out = run_watched(cmd, label, watch=[logf] + list(watch), env=env, cwd=cwd, log_file=logf, timeout=timeout, every=1, quiet=True)
    info = mon.stop()
    info.update(rc=rc, seconds=round(time.time() - t0), timed_out=timed_out, started=t0, ended=time.time(), gpu=gpu)
    return rc, last, timed_out, info


class GpuMonitor(Monitor):
    """The Monitor of step 1, for the GPU of one lane (nvidia-smi -i <n>): the base class would always read the first GPU."""

    def __init__(self, path, gpu, every=5):
        super().__init__(path, every)
        self.gpu = gpu

    def sample(self):
        gpu_mb, util = "", ""
        if shutil.which("nvidia-smi"):
            out = sh("nvidia-smi -i %d --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits" % self.gpu, timeout=20).splitlines()
            try:
                a, b = [x.strip() for x in out[0].split(",")]
                gpu_mb, util = int(a), int(b)
                self.peak_gpu_mb, self.peak_util = max(self.peak_gpu_mb, gpu_mb), max(self.peak_util, util)
            except (ValueError, IndexError):
                gpu_mb, util = "", ""
        ram = ""
        try:
            with open("/proc/meminfo") as f:
                for line in f:
                    if line.startswith("MemAvailable"):
                        ram = round(int(line.split()[1]) / 1048576, 2)
        except OSError:
            pass
        if ram != "":
            self.min_ram_gb = ram if self.min_ram_gb is None else min(self.min_ram_gb, ram)
        self.samples += 1
        with open(self.path, "a", encoding="utf-8") as f:
            f.write("%s,%s,%s,%s\n" % (datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M:%S"), gpu_mb, util, ram))


def save_state():
    """state.json from a snapshot: two workers change STATE while it is written, and json.dump refuses a dictionary that changes size."""
    for _ in range(5):
        try:
            text = json.dumps(STATE, indent=1, default=str)
        except RuntimeError:
            time.sleep(0.2)
            continue
        try:
            with open(STATE_FILE, "w", encoding="utf-8") as f:
                f.write(text)
        except OSError as ex:
            log("could not save state.json: %s" % ex)
        return
    log("could not save state.json: the state kept changing")


INPUTS = {}


def prepare_inputs(entry):
    """Map (unzipped), deposited model (for scoring) and the FASTA file of the entry; the same for every method."""
    if entry in INPUTS:
        return INPUTS[entry]
    row = {r["pdb_id"]: r for r in read_csv("data/entries_2026.csv")}.get(entry)
    if row is None:
        raise RuntimeError("%s is not in data/entries_2026.csv" % entry)
    folder = os.path.join(INPUT_ROOT, entry)
    os.makedirs(folder, exist_ok=True)
    number = row["emdb_id"].split("-")[-1]
    map_path = os.path.join(folder, "emd_%s.mrc" % number)
    if not os.path.exists(map_path):
        gz = download(row["map_url"], os.path.join(BIG, "downloads", os.path.basename(row["map_url"])), 100000)
        try:
            with gzip.open(gz, "rb") as src, open(map_path, "wb") as out:
                shutil.copyfileobj(src, out)
        except OSError:
            shutil.copyfile(gz, map_path)                           # not gzipped after all
    native = download(row["cif_url"], os.path.join(WORK, "native", entry + ".cif"), 1000)
    jobs = mica_inputs.read_jobs("data/af3/af3_jobs.csv", entry)
    fasta = os.path.join(folder, entry.lower() + ".fasta")
    with open(fasta, "w", encoding="utf-8") as f:
        f.write(mica_inputs.mica_fasta(jobs))
    info = {"entry": entry, "row": row, "folder": folder, "map": map_path, "map_sha256": sha256_file(map_path), "native": native, "jobs": jobs, "fasta": fasta}
    INPUTS[entry] = info
    log("%s: map %s (%d MB, sha256 %s), %d sequence(s): %s" % (entry, os.path.basename(map_path), os.path.getsize(map_path) // 1000000, info["map_sha256"][:16], len(jobs),
                                                              [{"job": r["job_name"], "length": int(r["length"]), "copies": int(r["copies_in_entry"])} for r in jobs]))
    return info


def af3_inputs(entry):
    """The AlphaFold Server prediction of every sequence of the entry (hashes checked against the list of the repository), in the order of the FASTA records."""
    info = prepare_inputs(entry)
    if "af3" not in info:
        jobs = info["jobs"]
        models = find_af3_dir(jobs)
        manifest = {}
        for line in open("data/af3/af3_files_sha256.txt", encoding="utf-8"):
            if line.strip():
                h, name = line.split()
                manifest[name] = h
        for r in jobs:
            name = "fold_%s_model_0.cif" % r["job_name"].lower()
            got = sha256_file(mica_inputs.find_model(models, r["job_name"]))
            if got != manifest.get(name):
                raise RuntimeError("the file %s is not the one that was checked (sha256 %s, expected %s)" % (name, got[:16], str(manifest.get(name))[:16]))
        built = mica_inputs.build("data/af3/af3_jobs.csv", models, entry, INPUT_ROOT)
        info["af3"] = [os.path.join(built["folder"], f["target"]) for f in built["files"]]
        log("AlphaFold Server files of %s: found in %s, hashes match the checked list" % (entry, models))
    return info["af3"]


def score_model(method, entry, row, native, kind, path):
    ev = evaluate_models.evaluate(STATE["usalign"], path, native, int(row["modeled_residues"]))
    os.makedirs(os.path.join(WORK, "scoring"), exist_ok=True)
    full = subprocess.run([STATE["usalign"], path, native, "-mol", "protein"], capture_output=True, text=True, errors="replace")
    with open(os.path.join(WORK, "scoring", "%s_%s_usalign.txt" % (entry, kind)), "w", encoding="utf-8") as f:
        f.write("# USalign %s %s -mol protein\n%s%s" % (path, native, full.stdout, full.stderr))
    return dict(ev, method=method, entry=entry, model_kind=kind, model_file=os.path.basename(path))


def keep_results(key, entry, variant, out_dir, produced):
    """Copies the models and the small text files of a run into the work folder (so they go into the zip); the intermediate files stay out."""
    for kind, path in produced.items():
        if kind in PRIVATE_KINDS:
            continue
        dst = os.path.join(WORK, "models", key, entry)
        os.makedirs(dst, exist_ok=True)
        shutil.copyfile(path, os.path.join(dst, os.path.basename(path)))
    dst = os.path.join(WORK, "method_files", key, entry, variant)
    for dirpath, _dirs, names in os.walk(out_dir):
        if os.path.relpath(dirpath, out_dir).count(os.sep) > 1:
            continue
        for n in names:
            p = os.path.join(dirpath, n)
            if n.endswith((".log", ".txt", ".csv", ".json", ".xlsx")) and os.path.getsize(p) <= 2000000:
                target = os.path.join(dst, os.path.relpath(p, out_dir))
                os.makedirs(os.path.dirname(target), exist_ok=True)
                shutil.copyfile(p, target)


def setup_method(spec):
    """Source at the pinned commit, the Python environment, a GPU test, then the method's own setup (weights); returns (python, source folder)."""
    key = spec["key"]
    repo = os.path.join(BIG, "src_" + key)
    log("source: %s at commit %s" % (spec["repo"], spec["commit"]))
    if not git_checkout(spec["repo"], spec["commit"], repo):
        raise RuntimeError("the source could not be fetched at commit %s (see the lines above)" % spec["commit"])
    env_py = make_env(spec, repo)
    if REQUIRE_GPU and not ENV_PY_GIVEN:
        gpu_test(spec, env_py)
    SETUPS[key](spec, repo, env_py)
    return env_py, repo


METRIC_COLUMNS = ["method", "entry", "model_kind", "model_file", "status", "note", "tm_score", "tm_score_pred_norm", "ca_rmsd", "aligned_ca_length", "seq_identity",
                  "pred_residues", "native_residues", "coverage"]
ROWS = []
SETUPS, PLANS, READY = {}, {}, {}


# ---- the time budget, the limits and the rules of the schedule
RUN_LIMIT = 10800                                           # seconds per run, the same for every method unless the spec of a method has its own `timeout` (LOG.md)
HOURS_SETTING = float(globals().get("HOURS", 9.0))
SESSION_S = min(HOURS_SETTING, 11.0) * 3600
RESERVE_S = float(os.environ.get("MICA_RESERVE_S", "600"))        # the end of the budget is kept for the summary and the zip
MIN_START_S = float(os.environ.get("MICA_MIN_START_S", "900"))    # a run is not started if less than this (or 1.25 times the usual time of its method) is left
POLL_S = float(os.environ.get("MICA_POLL", "10"))
GUARD_MINUTES = float(globals().get("GUARD_MINUTES", 60))
QUICK_CRASH_S = float(os.environ.get("MICA_QUICK_CRASH", GUARD_MINUTES * 60))  # a crash (not an out-of-memory kill) sooner than this is probably a problem of the installation
T0 = STATE["t0"]
HARD_STOP = T0 + SESSION_S - RESERVE_S                            # a program that still runs at this time is stopped ("interrupted", not a failure)
EXPLOSIVE = ('e3', 'emodelx')                                           # the pilot: AutoEM allocates np.zeros([6, N, N]) and the process is killed (out of memory); never two of them at once
TASKS = [['9T5K', 'cryozeta'], ['9WQ6', 'cryozeta'], ['22FX', 'cryozeta'], ['9V4D', 'cryozeta'], ['9JKK', 'cryozeta'], ['9OTA', 'cryozeta'], ['9TKA', 'cryozeta'], ['9KQM', 'cryozeta'], ['9WFL', 'cryozeta'], ['9WME', 'cryozeta'], ['9VI4', 'cryozeta'], ['9VBD', 'cryozeta'], ['9ZLH', 'cryozeta'], ['9RWS', 'cryozeta'], ['9VUH', 'cryozeta'], ['9WTX', 'cryozeta'], ['9OC0', 'cryozeta'], ['9Z24', 'cryozeta'], ['13BD', 'cryozeta'], ['26DK', 'cryozeta']]                                                   # [entry, method]: the order in which the tasks are taken
SCHED = threading.RLock()
INPUT_LOCK = threading.RLock()
PENDING = [tuple(t) for t in TASKS if t[1] in RUN_METHODS]
BUSY = {}                                                         # gpu -> what runs on it now
INTERVALS = []                                                    # (gpu, start, end) of the programs that finished
DURATIONS = {}                                                    # method -> seconds of its successful runs
SIZES = {'9T5K': 381, '22FX': 173, '9JKK': 530, '9TKA': 247, '9WFL': 682, '9VI4': 516, '9ZLH': 206, '9VUH': 634, '9OC0': 512, '13BD': 461, '9WQ6': 261, '9V4D': 276, '9OTA': 534, '9KQM': 403, '9WME': 446, '9VBD': 521, '9RWS': 512, '9WTX': 500, '9Z24': 338, '26DK': 191}                                                   # entry -> residues of the deposited model (data/entries_2026.csv)
RATES = {}                                                        # method -> seconds per residue of its successful runs (only for a method whose spec has `s_per_residue`)
CRASH_STREAK = {}
RETRY_ALONE = []
LANES = []
MAX_LANES = 2                                             # how many GPUs run work at the same time (1: one program at a time, for the entries that ran out of RAM next to another program)


def utc_iso(t):
    return datetime.datetime.fromtimestamp(t, datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def need_s(key, entry=None):
    """Seconds that must be left in the budget to start a run of this method (on this entry): 1.25 times its usual time, at least MIN_START_S. The usual time of a method is the mean of its
    successful runs in this session. For a method whose time grows with the size of the entry (`s_per_residue` in its spec) it is the residues of the entry times the largest seconds per
    residue seen in this session, or the rate of the pilot until a run has finished."""
    rate = SPECS[key].get("s_per_residue")
    if rate and entry in SIZES:
        seen = RATES.get(key) or []
        return max(MIN_START_S, 1.25 * (max(seen) if seen else rate) * SIZES[entry])
    d = DURATIONS.get(key) or []
    return max(MIN_START_S, 1.25 * sum(d) / len(d)) if d else MIN_START_S


def overlapped(gpu, t0, t1):
    """True if a program of another GPU ran at some time between t0 and t1."""
    with SCHED:
        spans = [(a, b) for g, a, b in INTERVALS if g != gpu] + [(b["program_start"], t1) for g, b in BUSY.items() if g != gpu and b.get("program_start")]
    return any(a < t1 and b > t0 for a, b in spans)


def mark(entry, key, status, note, failed_step=""):
    m = STATE["methods"][key]
    e = m["entries"].setdefault(entry, {"entry": entry, "method": key, "runs": {}, "models": []})
    e.update(status=status, failed_step=failed_step, note=note)


def disable_method(key, reason, status="disabled", failed_step=""):
    """No more tasks of this method in this session; its waiting tasks are 'not run' (they are not results: the next session runs them)."""
    with SCHED:
        m = STATE["methods"][key]
        m.update(status=status, failed_step=failed_step, note=reason)
        for t in [t for t in PENDING if t[1] == key]:
            PENDING.remove(t)
            mark(t[0], key, "not run", reason, failed_step)


def take_task(gpu):
    """The next task for this GPU: (entry, method), "wait" (something waits for the other GPU) or None (nothing more can start on this GPU)."""
    with SCHED:
        now = time.time()
        blocked = False
        for t in list(PENDING):
            entry, key = t
            if STATE["methods"][key]["status"] != "ready":
                continue
            if HARD_STOP - now < need_s(key, entry):
                continue                                                  # too late for this method (or this entry); a shorter one may still fit
            if key in EXPLOSIVE and any(b["key"] in EXPLOSIVE for g, b in BUSY.items() if g != gpu):
                blocked = True
                continue
            PENDING.remove(t)
            BUSY[gpu] = {"entry": entry, "key": key, "start": now, "program_start": None}
            return t
        return "wait" if blocked else None


def queue_line():
    done = sum(1 for m in STATE["methods"].values() for e in m["entries"].values() if e.get("status") == "ok")
    failed = sum(1 for m in STATE["methods"].values() for e in m["entries"].values() if e.get("status") == "failed")
    return "queue: %d waiting, %d ok, %d failed; %.1f of %.1f hours used" % (len(PENDING), done, failed, (time.time() - T0) / 3600, SESSION_S / 3600)


def run_task(gpu, entry, key, attempt=1):
    """One method on one entry on one GPU. Returns "ok", "failed", "quick_crash", "interrupted", "killed_corun", "inputs" or "time"."""
    spec, m = SPECS[key], STATE["methods"][key]
    env_py, repo = READY[key]
    first = m["entries"].get(entry) if attempt > 1 else None
    e = m["entries"][entry] = {"entry": entry, "method": key, "status": "not run", "failed_step": "", "note": "", "runs": {}, "models": [], "gpu": gpu, "attempt": attempt}
    if first:
        e["first_attempt"] = {"status": first.get("status"), "note": first.get("note"), "gpu": first.get("gpu"), "runs": first.get("runs")}
    tag = "GPU%d %s %s" % (gpu, spec["label"], entry)
    t_task = time.time()
    e["started_utc"] = utc_iso(t_task)
    log("\n" + "-" * 72 + "\n%s (attempt %d)\n%s\n" % (tag, attempt, queue_line()) + "-" * 72)
    try:
        with INPUT_LOCK:
            ctx = prepare_inputs(entry)
            runs = PLANS[key](spec, ctx, repo, env_py)
    except Exception as ex:
        e.update(status="not run", failed_step="inputs", note=str(ex)[:600])
        log("%s: NOT RUN, the inputs could not be prepared: %s" % (tag, ex))
        return "inputs"
    verdict = "ok"
    run_limit = spec.get("timeout") or RUN_LIMIT                          # the limit of one run of this method (the same for every method unless its spec says otherwise)
    for r in runs:
        limit = min(run_limit, HARD_STOP - time.time())
        if limit < min(60, MIN_START_S):
            e.update(status="not run", failed_step="", note="no time left in the session")
            return "time"
        cut = limit < run_limit - 1
        env = dict(r["env"], CUDA_VISIBLE_DEVICES=str(gpu))
        shutil.rmtree(r["out_dir"], ignore_errors=True)
        os.makedirs(r["out_dir"], exist_ok=True)
        log("\n=== %s, %s: %s" % (tag, r["variant"], " ".join(str(c) for c in r["cmd"][1:])[:600]))
        with SCHED:
            BUSY[gpu]["program_start"] = time.time()
        rc, last, timed_out, info = run_program(tag, r["cmd"], r["cwd"], env, "%s_%s_%s.log" % (entry, key, r["variant"]), gpu, watch=r.get("watch", ()), timeout=limit)
        with SCHED:
            INTERVALS.append((gpu, info["started"], info["ended"]))
            BUSY[gpu]["program_start"] = None
        info.update(limit_s=round(limit), other_gpu_busy=overlapped(gpu, info["started"], info["ended"]), started_utc=utc_iso(info["started"]), ended_utc=utc_iso(info["ended"]))
        e["runs"][r["variant"]] = info
        log("%s %s finished: exit code %s, %d s, GPU memory peak %s MB, lowest free RAM %s GB, other GPU busy: %s" % (
            tag, r["variant"], rc, info["seconds"], info["peak_gpu_mb"], info["min_ram_available_gb"], info["other_gpu_busy"]))
        produced = {k: p for k, p in r["outputs"].items() if os.path.exists(p) and os.path.getsize(p) > 500}
        main_ok = r["main"] in produced
        if timed_out and cut:
            keep_results(key, entry, r["variant"], r["out_dir"], {})
            e.update(status="interrupted", failed_step=r["variant"], note="stopped by the end of the session after %d s; run again in the next session" % info["seconds"])
            log("%s: INTERRUPTED by the end of the session (not a failure)" % tag)
            pack_now()
            return "interrupted"
        if rc == -9 and not timed_out and info["other_gpu_busy"] and attempt == 1:
            keep_results(key, entry, r["variant"], r["out_dir"], {})
            e.update(status="killed_corun", failed_step=r["variant"], note="killed by the system (exit code -9, probably out of memory) while the other GPU's program ran; run again alone")
            log("%s: KILLED while the other GPU ran; it is run again alone at the end" % tag)
            RETRY_ALONE.append((entry, key))
            pack_now()
            return "killed_corun"
        keep_results(key, entry, r["variant"], r["out_dir"], produced)
        if rc != 0 or timed_out or not main_ok:
            gpu_oom = bool(spec.get("oom_is_result")) and any("out of memory" in line.lower() for line in last)    # CryoAtom: 15 GB of GPU memory may be too little for an entry; that is a result
            crashed = (rc not in (0, -9) or (rc == 0 and not main_ok)) and not timed_out and not gpu_oom          # an out-of-memory kill (-9) is a result, a crash is a suspect
            verdict = "quick_crash" if (crashed and info["ended"] - info["started"] < QUICK_CRASH_S) else "failed"
            if timed_out:
                note = "timeout: the run limit of %d s" % limit
            elif rc == -9:
                note = "killed by the system (exit code -9, probably out of memory%s)" % ("; also when it ran alone" if attempt > 1 else "")
            elif gpu_oom:
                note = "out of GPU memory (exit code %s; the error message of the program)" % rc
            else:
                note = "exit code %s" % rc
            if not main_ok:
                note += "; the model file %s was not written" % os.path.basename(r["outputs"][r["main"]])
            e.update(status="failed", failed_step=r["variant"], note=note)
            log("%s: FAILED: %s. The last lines of the program:\n%s" % (tag, note, "\n".join(last[-20:])))
        log("%s wrote: %s" % (spec["label"], ", ".join("%s (%d bytes)" % (os.path.relpath(os.path.join(d, n), r["out_dir"]), os.path.getsize(os.path.join(d, n)))
                                                         for d, _s, ns in os.walk(r["out_dir"]) for n in sorted(ns))[:700]))
        for kind, path in produced.items():
            try:
                row = score_model(key, entry, ctx["row"], ctx["native"], kind, path)
                ROWS.append(row)
                e["models"].append(row)
            except Exception as ex:
                e.setdefault("score_errors", []).append("%s: %s" % (kind, str(ex)[:200]))
                log("scoring of %s stopped: %s (the model is in the zip; it is scored again later)" % (kind, ex))
        pack_now()
    e["seconds"] = round(time.time() - t_task)
    if verdict == "ok":
        e["status"] = "ok"
        spent = sum(v["seconds"] for v in e["runs"].values())
        DURATIONS.setdefault(key, []).append(spent)
        if spec.get("s_per_residue") and entry in SIZES:
            RATES.setdefault(key, []).append(spent / SIZES[entry])
    return verdict


def after_task(entry, key, result):
    """The rules that look at the outcome: two crashes within three minutes in a row stop the method for this session."""
    with SCHED:
        if result == "quick_crash":
            CRASH_STREAK[key] = CRASH_STREAK.get(key, 0) + 1
        elif result in ("ok", "failed"):
            CRASH_STREAK[key] = 0
        streak = CRASH_STREAK.get(key, 0)
    if streak >= 2 and STATE["methods"][key]["status"] == "ready":
        reason = ("two runs in a row ended in a crash within %d minutes: probably a problem of the installation or of the inputs, not a result; "
                  "the other entries of this method are left for the next session" % (QUICK_CRASH_S // 60))
        log("\n%s: %s" % (SPECS[key]["label"], reason))
        disable_method(key, reason, status="disabled", failed_step="guard")


def lane_main(gpu):
    """The worker of one GPU: takes the next task until nothing more can start."""
    while True:
        t = take_task(gpu)
        if t is None:
            log("GPU %d: nothing more to start in this session" % gpu)
            return
        if t == "wait":
            time.sleep(POLL_S)
            continue
        entry, key = t
        try:
            result = run_task(gpu, entry, key)
        except Exception:
            import traceback
            result = "quick_crash"
            m = STATE["methods"][key]
            m["entries"].setdefault(entry, {"entry": entry, "method": key, "runs": {}, "models": []}).update(
                status="not run", failed_step="notebook error", note=traceback.format_exc()[-600:])
            log("\nGPU %d: %s on %s STOPPED by an error of the notebook (not a result of the method):\n%s" % (gpu, SPECS[key]["label"], entry, traceback.format_exc()))
        finally:
            with SCHED:
                BUSY.pop(gpu, None)
        after_task(entry, key, result)


def detect_gpus():
    forced = os.environ.get("MICA_TEST_GPUS")                         # only the tests set this
    if forced:
        return list(range(int(forced)))
    ids = []
    if shutil.which("nvidia-smi"):
        for line in sh("nvidia-smi --query-gpu=index --format=csv,noheader", timeout=30).splitlines():
            if line.strip().isdigit():
                ids.append(int(line.strip()))
    return ids or [0]


def usable_lanes():
    """The GPUs that run work: every GPU that a small PyTorch computation can use (tested with the environment of the first method that is ready)."""
    gpus = detect_gpus()
    if len(gpus) < 2 or ENV_PY_GIVEN or not REQUIRE_GPU:
        return gpus
    ready = [READY[k][0] for k in RUN_METHODS if k in READY]
    if not ready:
        return gpus[:1]
    good = []
    for g in gpus:
        rc, last, _ = run([ready[0], "-c", GPU_TEST], env=env_vars(ready[0], {"CUDA_VISIBLE_DEVICES": str(g)}), quiet=True)
        log("GPU %d: %s" % (g, "a PyTorch computation works" if rc == 0 else "does NOT work (exit code %s): %s; it is not used" % (rc, " ".join(last[-2:])[:200])))
        if rc == 0:
            good.append(g)
    return good or gpus[:1]


def check_af3_files():
    """Stops the session at once, before an hour is spent, if the private AlphaFold dataset is not attached or is not the checked one (the hashes of the list of the repository)."""
    entries = sorted({t[0] for t in PENDING if SPECS[t[1]]["uses_af3"]})
    if not entries:
        return
    manifest = {}
    with open("data/af3/af3_files_sha256.txt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                h, name = line.split()
                manifest[name] = h
    models = ""
    for entry in entries:
        jobs = mica_inputs.read_jobs("data/af3/af3_jobs.csv", entry)
        models = find_af3_dir(jobs)
        for r in jobs:
            name = "fold_%s_model_0.cif" % r["job_name"].lower()
            got = sha256_file(mica_inputs.find_model(models, r["job_name"]))
            if got != manifest.get(name):
                raise RuntimeError("the file %s is not the one that was checked (sha256 %s, expected %s)" % (name, got[:16], str(manifest.get(name))[:16]))
    log("AlphaFold Server files of %d entries: found in %s, every hash matches the checked list" % (len(entries), models))


def setup_phase():
    """Source, environment, weights of every method, one after the other; a method whose setup fails is left out (its tasks are 'not run'), the others go on."""
    for key in RUN_METHODS:
        spec = SPECS[key]
        m = STATE["methods"][key] = {"key": key, "label": spec["label"], "commit": spec["commit"], "status": "setup", "failed_step": "", "note": "", "setup_seconds": None, "entries": {}}
        log("\n" + "#" * 78 + "\n# setup of %s (commit %s)\n" % (spec["label"], spec["commit"][:10]) + "#" * 78)
        t0 = time.time()
        try:
            READY[key] = setup_method(spec)
            m.update(status="ready", setup_seconds=round(time.time() - t0))
            log("%s is ready (%d s)" % (spec["label"], m["setup_seconds"]))
        except Exception as ex:
            import traceback
            m["setup_seconds"] = round(time.time() - t0)
            log("\n%s STOPPED in its setup: %s\n%s" % (spec["label"], ex, traceback.format_exc()[-1200:]))
            disable_method(key, "the setup failed: %s" % str(ex)[:400], status="setup failed", failed_step="setup")
        pack_now()


log("tools ready; methods: %s; %d tasks; budget %.1f hours (the programs are stopped %.0f minutes before the end)" % (", ".join(RUN_METHODS), len(PENDING), SESSION_S / 3600, RESERVE_S / 60))


## Step 4. US-align

US-align is compiled from its pinned commit (the program that scores every model).

In [ ]:
if STATE["ok"]:
    build_usalign()
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Step 5. CryoZeta

The four programs of the authors' `inference_demo.sh`, in the same order and with the same settings (5 samples, 20 diffusion steps, 10 cycles, seed 101): `cryozeta-detection json-run`,
`cryozeta-inference`, `cryozeta-inference --use_interpolation true` and `cryozeta-combine --num-select 5`. The model of the benchmark is `<entry>_sample_0.cif` of `CryoZeta-Final`, the sample that the program
ranks first by its own confidence; the four other samples are kept and scored too. A small program of the repository (`scripts/cryozeta_tools.py`, standard library only) runs the four steps for
one entry because `cryozeta-inference` catches the error of a sample, writes it to a file and ends with exit code 0: it looks at the files and not at the exit code.

**What is installed.** The environment `cu11` of the project (Python 3.12, PyTorch 2.4 with CUDA 11.8, the one the project selects for a GPU of the Turing generation), exactly as the lock file of the
repository says (`pixi install -e cu11 --frozen`, pixi 0.41.1). TEASER++ is built at a pinned commit with pinned `cmake` and `pybind11` from PyPI (the project builds its master). The weights are the three
files `cryozeta*-v0.0.1.safetensors` of the Hugging Face repository `KiharaLab/CryoZeta` at the revision that the Hub reports; their sha256 is compared with the one of the Hub.
**Input.** The map, the resolution of the entries table, the recommended contour level of the EMDB header, the sequences and numbers of copies of the deposited model, and for every chain the alignment from the public
ColabFold server (`api.colabfold.com`, UniRef30 and the environmental databases; the README of CryoZeta names the ColabFold search scripts as the way to get MSAs). The server's alignments carry no species,
so no chains are paired across sequences; that is a limit of this input and it is reported.
**Settings that depend on this machine.** The compiled kernels of the demo script (DeepSpeed and cuEquivariance attention, the fast layer norm) need a CUDA compiler and an Ampere GPU: they are off (the
program has a plain implementation of each). The attention chunk size starts at the default (256) and goes down to 64, 16 and 4 only after an out-of-memory error. The number format is the default (bfloat16);
if a stage fails for another reason it is tried once in float32. If the GPU test of the setup shows that this GPU cannot run bfloat16 in the detection network, two lines of the detection code are changed to
float32 (`cryozeta_source_patch.diff` in the zip shows what was changed; it is empty if nothing was).

In [ ]:
import cryozeta_tools as CZT

CZ = {}                                   # what the setup found: the folders of TEASER++ and of the weights, the number format, the result of the GPU test
CZ_PROGRAMS = ("cryozeta-detection", "cryozeta-inference", "cryozeta-combine")
CZ_PREFLIGHT = r"""
import json, torch
out = {"torch": torch.__version__, "cuda": torch.version.cuda, "cuda_available": torch.cuda.is_available()}
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    out.update(gpu=p.name, memory_gb=round(p.total_memory / 2**30, 2), capability="%d.%d" % (p.major, p.minor))

    def attempt(name, fn):
        try:
            fn()
            torch.cuda.synchronize()
            out[name] = "ok"
        except Exception as ex:
            out[name] = "failed: " + (str(ex).splitlines() or [type(ex).__name__])[0][:160]

    attempt("bf16_matmul", lambda: torch.randn(256, 256, device="cuda", dtype=torch.bfloat16) @ torch.randn(256, 256, device="cuda", dtype=torch.bfloat16))
    x = torch.randn(1, 4, 16, 16, 16, device="cuda")
    conv = torch.nn.Conv3d(4, 8, 3, padding=1).cuda()
    conv16 = torch.nn.Conv3d(4, 8, 3, padding=1).cuda().to(torch.bfloat16)
    attempt("bf16_conv3d", lambda: conv16(x.to(torch.bfloat16)))

    def amp():
        with torch.autocast("cuda", dtype=torch.bfloat16):
            conv(x)

    attempt("bf16_autocast_conv3d", amp)
print(json.dumps(out))
"""
_env_vars_plain = env_vars


def env_vars(env_py, extra=None):
    """The environment of a program of a method (the function of the tools), and for a conda prefix (the pixi environment of CryoZeta) what its activation script sets: CONDA_PREFIX and its lib folder
    on LD_LIBRARY_PATH."""
    e = _env_vars_plain(env_py, extra)
    prefix = os.path.dirname(os.path.dirname(env_py))
    if os.path.isdir(os.path.join(prefix, "conda-meta")):
        e.setdefault("CONDA_PREFIX", prefix)
        old = os.environ.get("LD_LIBRARY_PATH", "")
        e.setdefault("LD_LIBRARY_PATH", os.path.join(prefix, "lib") + (os.pathsep + old if old else ""))
    return e


def cz_prefix(env_py):
    return os.path.dirname(os.path.dirname(env_py))


def cz_env(prefix, teaser=None):
    """What `env_setup.sh` of the project sets when pixi activates the environment (the activation script itself needs pixi), for the programs that the notebook starts."""
    e = env_vars(os.path.join(prefix, "bin", "python"), {"CUDA_DEVICE_ORDER": "PCI_BUS_ID", "USE_OPM_CHUNKED": "1", "CUTLASS_PATH": prefix})
    if teaser:
        e["TEASERPP_DIR"] = teaser
        e["PYTHONPATH"] = os.path.join(teaser, "build", "python")
    return e


def pixi_vars():
    return {"PIXI_HOME": os.path.join(BIG, "pixi_home"), "PIXI_CACHE_DIR": os.path.join(BIG, "pixi_cache"), "RATTLER_CACHE_DIR": os.path.join(BIG, "rattler_cache"),
            "UV_CACHE_DIR": os.environ.get("UV_CACHE_DIR") or os.path.join(BIG, "uv_cache"), "PIXI_NO_PROGRESS": "true", "PIXI_COLOR": "never", "NO_COLOR": "1"}


def install_pixi(spec):
    """The pinned release of pixi, the program that installs the environment of CryoZeta from the lock file of the project."""
    p = spec["pixi"]
    exe = os.path.join(BIG, "pixi", "pixi-%s" % p["version"])
    if not (os.path.exists(exe) and os.access(exe, os.X_OK)):
        if not fetch(p["url"], exe, p["min_bytes"], "pixi %s" % p["version"]):
            raise RuntimeError("pixi could not be downloaded from %s" % p["url"])
        if p.get("sha256") and sha256_file(exe) != p["sha256"]:
            got = sha256_file(exe)
            os.remove(exe)
            raise RuntimeError("the downloaded pixi does not have the sha256 of release %s (%s, expected %s)" % (p["version"], got[:16], p["sha256"][:16]))
        os.chmod(exe, 0o755)
    rc, last, _ = run([exe, "--version"], env=pixi_vars(), quiet=True)
    if rc != 0 or p["version"] not in " ".join(last):
        raise RuntimeError("the downloaded pixi does not report version %s (%s)" % (p["version"], " ".join(last[-2:])[:200]))
    log("pixi: " + last[-1])
    return exe


def make_env_cryozeta(spec, repo):
    """The conda environment of CryoZeta, installed exactly as the lock file of the repository says; returns the path of its python."""
    name = spec["pixi"]["env"]
    env_py = os.path.join(repo, ".pixi", "envs", name, "bin", "python")
    test = (IMPORT_TESTS or {}).get(spec["key"]) or spec["import_test"]
    if os.path.exists(env_py):
        rc, last, _ = run([env_py, "-c", test], env=env_vars(env_py), cwd=repo, quiet=True)
        if rc == 0:
            log("the environment of %s exists already: %s" % (spec["label"], last[-1]))
            return env_py
    for f in ("pixi.lock", "pyproject.toml", "env_setup.sh"):
        if not os.path.exists(os.path.join(repo, f)):
            raise RuntimeError("the CryoZeta source at commit %s lacks %s" % (spec["commit"], f))
    pixi = install_pixi(spec)
    logf = os.path.join(WORK, "logs", "cryozeta_pixi_install.log")
    log("installing the environment of %s with pixi, from the lock file of the repository (10 to 25 minutes) ..." % spec["label"])
    rc, last, timed_out = run_watched([pixi, "install", "-e", name, "--frozen"], "pixi install", watch=[logf], env=pixi_vars(), cwd=repo, log_file=logf, timeout=3600, every=1, quiet=True)
    if rc != 0 or timed_out or not os.path.exists(env_py):
        log("\n".join(last[-25:]))
        raise RuntimeError("pixi could not install the environment %s (%s; see logs/cryozeta_pixi_install.log in the zip)" % (name, "timeout" if timed_out else "exit code %s" % rc))
    rc, last, _ = run([pixi, "list", "-e", name, "--no-install"], env=pixi_vars(), cwd=repo, quiet=True)
    with open(os.path.join(WORK, "cryozeta_pixi_list.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(last) if rc == 0 else "pixi list failed (exit code %s)\n%s" % (rc, "\n".join(last)))
    rc, last, _ = run([env_py, "-c", test], env=env_vars(env_py), cwd=repo)
    if rc != 0:
        raise RuntimeError("the packages of CryoZeta were installed but its own modules do not import (see the lines above)")
    log(last[-1])
    return env_py


_make_env_plain = make_env


def make_env(spec, repo):
    """The environment of a method: the one of the tools (uv, pinned wheels) for every method but CryoZeta, whose project is a pixi project."""
    if spec["key"] != "cryozeta":
        return _make_env_plain(spec, repo)
    if ENV_PY_GIVEN:
        return ENV_PY_GIVEN
    return make_env_cryozeta(spec, repo)


def build_teaser(spec, prefix):
    """TEASER++ (the registration library that CryoZeta imports as teaserpp_python) at a pinned commit, built with the headers of the environment (Eigen, Boost, Python), the compiler of this
    machine and pinned cmake and pybind11 (PyPI; the lock file has neither); returns the folder. The project's own task builds the master branch with the cmake of the system."""
    t = spec["teaser"]
    if t.get("prebuilt"):                                                     # the tests
        return t["prebuilt"]
    root = os.path.join(BIG, "teaser_plusplus")
    if not git_checkout(t["repo"], t["commit"], root):
        raise RuntimeError("TEASER++ could not be fetched at commit %s (see the lines above)" % t["commit"])
    build = os.path.join(root, "build")
    pkg = os.path.join(build, "python", "teaserpp_python")
    if not glob.glob(os.path.join(pkg, "_teaserpp*.so")):
        tools = os.path.join(BIG, "teaser_tools")
        cmake = os.path.join(tools, "cmake", "data", "bin", "cmake")
        pyb = os.path.join(tools, "pybind11", "share", "cmake", "pybind11")
        if not (os.path.exists(cmake) and os.path.isdir(pyb)):
            rc, last, _ = run([sys.executable, "-m", "pip", "install", "-q", "--target", tools, "cmake==" + t["cmake"], "pybind11==" + t["pybind11"]], quiet=True)
            if rc != 0 or not os.path.exists(cmake):
                log("\n".join(last[-12:]))
                raise RuntimeError("pip could not install cmake %s and pybind11 %s for the TEASER++ build" % (t["cmake"], t["pybind11"]))
        shutil.rmtree(build, ignore_errors=True)
        logf = os.path.join(WORK, "logs", "cryozeta_teaser_build.log")
        py = os.path.join(prefix, "bin", "python")
        steps = (("configure", [cmake, "-S", root, "-B", build, "-DCMAKE_BUILD_TYPE=Release", "-DBUILD_PYTHON_BINDINGS=ON", "-DCMAKE_INTERPROCEDURAL_OPTIMIZATION=OFF", "-DBUILD_TESTS=OFF",
                                "-DBUILD_DOC=OFF", "-DCMAKE_PREFIX_PATH=" + prefix, "-Dpybind11_DIR=" + pyb, "-DPYTHON_EXECUTABLE=" + py]),
                 ("build", [cmake, "--build", build, "-j", str(max(1, os.cpu_count() or 1))]))
        for what, cmd in steps:
            log("TEASER++: %s ..." % what)
            rc, last, timed_out = run_watched(cmd, "TEASER++ " + what, watch=[logf], env={"CC": None, "CXX": None, "PYTHONPATH": None}, log_file=logf, timeout=3000, every=1, quiet=True)
            if rc != 0 or timed_out:
                log("\n".join(last[-25:]))
                raise RuntimeError("TEASER++ could not be built (%s: %s; see logs/cryozeta_teaser_build.log in the zip)" % (what, "timeout" if timed_out else "exit code %s" % rc))
        if not glob.glob(os.path.join(pkg, "_teaserpp*.so")):
            raise RuntimeError("the TEASER++ build did not make the Python module (nothing like %s)" % os.path.join(pkg, "_teaserpp*.so"))
    init = os.path.join(root, "python", "teaserpp_python", "__init__.py")
    if os.path.exists(init) and not os.path.exists(os.path.join(pkg, "__init__.py")):
        shutil.copyfile(init, os.path.join(pkg, "__init__.py"))               # as env_setup.sh does: the build makes the compiled module only
    return root


def fetch_cryozeta_weights(spec):
    """The three weights files from the Hugging Face repository, at the revision that the Hub reports, with the size and the sha256 that the Hub lists for them; returns (folder, facts, revision)."""
    import urllib.request
    hub = spec["hub"]
    try:
        with urllib.request.urlopen(urllib.request.Request(hub["api"], headers={"User-Agent": "mica-benchmark-cryozeta"}), timeout=60) as r:
            meta = json.loads(r.read().decode("utf-8", "replace"))
    except Exception as ex:
        raise RuntimeError("the file list of %s could not be read from %s: %s" % (hub["repo"], hub["api"], str(ex)[:200]))
    rev = meta.get("sha") or "main"
    sib = {s["rfilename"]: s for s in meta.get("siblings", []) if s.get("rfilename")}
    with open(os.path.join(WORK, "cryozeta_hub_files.json"), "w", encoding="utf-8") as f:
        json.dump({"repo": hub["repo"], "revision": rev, "last_modified": meta.get("lastModified", ""),
                   "files": [{"path": k, "size": v.get("size"), "lfs_sha256": (v.get("lfs") or {}).get("sha256", "")} for k, v in sorted(sib.items())]}, f, indent=1)
    assets = os.path.join(BIG, "cryozeta_assets")
    facts = {}
    for role, fname in hub["files"].items():
        paths = [k for k in sib if k == fname] or [k for k in sib if os.path.basename(k) == fname]
        if not paths:
            raise RuntimeError("the Hugging Face repository %s has no file %s at revision %s (it has %s)" % (hub["repo"], fname, rev[:10], sorted(sib)[:12]))
        meta_f = sib[paths[0]]
        lfs = meta_f.get("lfs") or {}
        size = meta_f.get("size") or lfs.get("size")
        dest = os.path.join(assets, fname)
        if not fetch(hub["resolve"].format(rev=rev, path=paths[0]), dest, hub["min_bytes"], "CryoZeta weights " + fname, expect_size=size):
            raise RuntimeError("the weights file %s could not be downloaded from %s" % (fname, hub["repo"]))
        digest = sha256_file(dest)
        if lfs.get("sha256") and digest != lfs["sha256"]:
            raise RuntimeError("the weights file %s does not have the sha256 that the Hub lists (%s, expected %s)" % (fname, digest[:16], lfs["sha256"][:16]))
        facts[role] = {"file": fname, "hub_path": paths[0], "bytes": os.path.getsize(dest), "sha256": digest, "hub_sha256": lfs.get("sha256", "")}
        log("%s: %d MB, sha256 %s%s" % (fname, facts[role]["bytes"] // 1000000, digest[:16], " (the Hub's value)" if lfs.get("sha256") else " (the Hub lists none)"))
    return assets, facts, rev


def cryozeta_preflight(spec, repo, env_py, prefix):
    """Before any hour is spent: the modules of the three programs import together with TEASER++, and (on a GPU) bfloat16 works. A GPU that cannot run bfloat16 in the detection network gets
    the two-line change of the detection code (float32) and a float32 start; nothing is changed on a GPU that can."""
    env = cz_env(prefix, CZ["teaser"])
    code = "import cryozeta.runner.inference_detection, cryozeta.runner.inference, cryozeta.runner.combine, teaserpp_python; print('the modules of the three programs and TEASER++ import')"
    if not ENV_PY_GIVEN:
        rc, last, _ = run([env_py, "-c", code], env=env, cwd=repo)
        if rc != 0:
            raise RuntimeError("the modules of the CryoZeta programs do not import together with TEASER++ (see the lines above)")
    pf, patched = {}, ""
    if REQUIRE_GPU and not ENV_PY_GIVEN:
        rc, last, _ = run([env_py, "-c", CZ_PREFLIGHT], env=env, cwd=repo, quiet=True)
        try:
            pf = json.loads(last[-1])
        except (ValueError, IndexError):
            raise RuntimeError("the GPU test of the CryoZeta environment gave no result (exit code %s): %s" % (rc, " ".join(last[-3:])[:300]))
        log("GPU test of the CryoZeta environment: %s" % json.dumps(pf))
        if not pf.get("cuda_available"):
            raise RuntimeError("PyTorch in the CryoZeta environment does not see the GPU")
        if pf.get("bf16_conv3d") != "ok" or pf.get("bf16_autocast_conv3d") != "ok":
            patched = CZT.patch_detection_precision(repo)
            log("this GPU cannot run the detection network in bfloat16 (%s): two lines of the detection code are changed to float32:\n%s" % (pf.get("bf16_conv3d"), patched))
    with open(os.path.join(WORK, "cryozeta_source_patch.diff"), "w", encoding="utf-8") as f:
        f.write(sh("git -C %s diff" % repo, timeout=60) + "\n")
    CZ["dtype"] = spec["run"]["dtype"] if (not pf or pf.get("bf16_matmul") == "ok") else "fp32"
    CZ["preflight"] = pf
    STATE["methods"][spec["key"]]["preflight"] = {"gpu_test": pf, "detection_patched": bool(patched), "dtype_start": CZ["dtype"]}
    if CZ["dtype"] != spec["run"]["dtype"]:
        log("bfloat16 matrix products do not work on this GPU: the runs start in float32")


def setup_cryozeta(spec, repo, env_py):
    """TEASER++, the weights and the checks on this GPU; the environment is the one that make_env made."""
    prefix = cz_prefix(env_py)
    missing = [p for p in CZ_PROGRAMS if not os.path.exists(os.path.join(prefix, "bin", p))]
    if missing:
        raise RuntimeError("the environment lacks the programs %s (the project's console scripts)" % ", ".join(missing))
    m = STATE["methods"][spec["key"]]
    CZ["teaser"] = build_teaser(spec, prefix)
    CZ["assets"], files, rev = fetch_cryozeta_weights(spec)
    m["weights_hub"] = {"repo": spec["hub"]["repo"], "revision": rev, "files": files}
    cryozeta_preflight(spec, repo, env_py, prefix)
    m["version"] = "commit %s, pixi %s, environment %s, weights revision %s" % (spec["commit"][:10], spec["pixi"]["version"], spec["pixi"]["env"], rev[:10])
    m["teaser"] = {"commit": spec["teaser"]["commit"], "folder": CZ["teaser"]}
    record_files(spec["key"], [CZ["assets"]])


def plan_cryozeta(spec, ctx, repo, env_py):
    """One run per entry: the runner of the repository (scripts/cryozeta_tools.py) with a settings file; it fetches the contour level and the alignments, runs the four programs of the
    authors' demo and writes the summary of what happened."""
    entry, row = ctx["entry"], ctx["row"]
    out = os.path.join(RUN_ROOT, "cryozeta", entry)
    inp = os.path.join(INPUT_ROOT, entry)
    run_s = spec["run"]
    cfg = {"name": entry, "repo": repo, "env_dir": cz_prefix(env_py), "teaser_dir": CZ["teaser"], "assets": CZ["assets"], "weights": spec["hub"]["files"],
           "map_path": ctx["map"], "resolution": float(row["resolution_A"]), "header_url": CZT.emdb_header_url(row["map_url"]), "colabfold_host": spec["colabfold"],
           "chains": [{"sequence": j["sequence_submitted"].strip(), "count": int(j["copies_in_entry"])} for j in ctx["jobs"]], "msa_root": os.path.join(inp, "cryozeta_msa"), "msa_poll_s": float(spec["msa_poll_s"]), "msa_tries": int(spec["msa_tries"]),
           "input_json": os.path.join(inp, "cryozeta_input.json"), "dump_dir": out, "seed": run_s["seed"], "n_sample": run_s["n_sample"], "n_step": run_s["n_step"], "n_cycle": run_s["n_cycle"],
           "num_select": run_s["num_select"], "dtype": CZ.get("dtype", run_s["dtype"]), "kernels": run_s["kernels"], "chunk_sizes": list(CZT.CHUNK_LADDER),
           "summary_path": os.path.join(out, "driver_summary.json"), "progress_path": os.path.join(out, "progress.txt")}
    os.makedirs(inp, exist_ok=True)
    cfg_path = os.path.join(inp, "cryozeta_config.json")
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=1)
    final = os.path.join(out, entry, "CryoZeta-Final")
    outputs = {("cryozeta" if i == 0 else "cryozeta_s%d" % i): os.path.join(final, "%s_sample_%d.cif" % (entry, i)) for i in range(int(run_s["num_select"]))}
    cmd = [sys.executable, os.path.join(WORK, "scripts", "cryozeta_tools.py"), "run", cfg_path]
    # the runner itself is plain Python of this machine: it gets the environment of any method (not the library path of the conda environment, which its own programs get from the runner)
    return [{"variant": "run", "cmd": cmd, "cwd": repo, "env": _env_vars_plain(env_py), "out_dir": out, "outputs": outputs, "main": "cryozeta", "watch": [cfg["progress_path"]]}]


SETUPS["cryozeta"], PLANS["cryozeta"] = setup_cryozeta, plan_cryozeta


## Step 6. The runs

First the setup of CryoZeta (source, environment, TEASER++, weights, checks on this GPU), then one worker per GPU takes entries from the list until the time budget says stop. A progress line of every running
program appears every two minutes (the stage that the runner is in and its last line); the output of the programs is in `logs/` of the zip. Everything is packed into a zip after every run.

In [ ]:
def run_schedule():
    global LANES
    setup_phase()
    LANES = usable_lanes()[:MAX_LANES]
    log("\nGPUs used: %s; %s" % (LANES, queue_line()))
    workers = [threading.Thread(target=lane_main, args=(g,), name="gpu%d" % g) for g in LANES]
    for w in workers:
        w.start()
    for w in workers:
        w.join()
    for entry, key in list(RETRY_ALONE):                      # killed while the other GPU ran: once more, alone
        if STATE["methods"][key]["status"] == "ready" and HARD_STOP - time.time() >= need_s(key, entry):
            log("\nrun again alone: %s on %s" % (SPECS[key]["label"], entry))
            with SCHED:
                BUSY[LANES[0]] = {"entry": entry, "key": key, "start": time.time(), "program_start": None}
            try:
                run_task(LANES[0], entry, key, attempt=2)
            finally:
                with SCHED:
                    BUSY.pop(LANES[0], None)
        else:
            log("\n%s on %s cannot be run again alone now (no time left or the method was stopped): it stays 'killed_corun' and is run again in the next session" % (SPECS[key]["label"], entry))
    for entry, key in list(PENDING):
        PENDING.remove((entry, key))
        mark(entry, key, "not run", "not started: not enough time left in the session")


if STATE["ok"] and STATE["usalign"]:
    try:
        check_af3_files()
    except Exception as ex:
        fail("the private AlphaFold dataset is missing or is not the checked one (%s). Kaggle: right-hand panel, Add Input, choose your AlphaFold dataset, then run again. "
             "No GPU time was spent." % str(ex)[:300])
    if STATE["ok"]:
        try:
            run_schedule()
        except BaseException as ex:
            import traceback
            fail("the schedule stopped with an error of the notebook: %s" % str(ex)[:300])
            log(traceback.format_exc())
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Step 7. Summary

`metrics_usalign.csv` (every model that was scored, first pass: the final scores are computed later from the models in the zip), `method_status.csv` (one line per task),
`pending.csv` (the tasks for the next session), and a table of the whole run.

In [ ]:
def status_rows():
    rows = []
    for key in RUN_METHODS:
        m = STATE["methods"].get(key)
        if not m:
            continue
        for entry, e in m["entries"].items():
            runs = list(e.get("runs", {}).values())
            rows.append(dict(method=key, entry=entry, status=e.get("status", ""), failed_step=e.get("failed_step", ""), note=e.get("note", ""), attempt=e.get("attempt", ""),
                             gpu=e.get("gpu", ""), started_utc=e.get("started_utc", ""), seconds=sum(r["seconds"] for r in runs) if runs else "",
                             peak_gpu_mb=max([r["peak_gpu_mb"] for r in runs] or [""]) if runs else "", min_ram_gb=min([r["min_ram_available_gb"] for r in runs if r["min_ram_available_gb"] is not None] or [""]) if runs else "",
                             other_gpu_busy=any(r["other_gpu_busy"] for r in runs) if runs else "", limit_s=runs[0]["limit_s"] if runs else ""))
    return rows


def summary_table(rows):
    keys = [k for k in RUN_METHODS if k in STATE["methods"]]
    entries = [t[0] for t in TASKS]
    entries = list(dict.fromkeys(entries))
    by = {(r["entry"], r["method"]): r for r in rows}
    best = {}
    for r in ROWS:
        if r["model_kind"] == SPECS[r["method"]]["main_kind"]:
            best[(r["entry"], r["method"])] = r["tm_score"]
    log("\n%-6s " % "entry" + " ".join("%-24s" % SPECS[k]["label"][:24] for k in keys))
    for entry in entries:
        cells = []
        for k in keys:
            r = by.get((entry, k))
            if not r:
                cells.append("%-24s" % "-")
            else:
                mins = ("%dmin" % round(r["seconds"] / 60)) if r["seconds"] != "" else ""
                cells.append("%-24s" % ("%s %s %s" % (r["status"], mins, ("TM " + str(best[(entry, k)])) if (entry, k) in best else "")).strip()[:24])
        log("%-6s " % entry + " ".join(cells))
    log("")
    for k in keys:
        m = STATE["methods"][k]
        log("%-14s %-13s setup %s s%s" % (SPECS[k]["label"], m["status"].upper(), m["setup_seconds"], ("; " + m["note"][:160]) if m["status"] != "ready" else ""))


for _k, _s in SPECS.items():
    _s["main_kind"] = {"modelangelo": "modelangelo", "e3": "e3_prealign", "emprot": "emprot", "emodelx": "emodelx_af_all_atom", "cryoatom1": "cryoatom1", "cryozeta": "cryozeta"}[_k]
if STATE["ok"]:
    rows = status_rows()
    write_csv("metrics_usalign.csv", METRIC_COLUMNS, ROWS)
    write_csv("method_status.csv", ["method", "entry", "status", "failed_step", "note", "attempt", "gpu", "started_utc", "seconds", "peak_gpu_mb", "min_ram_gb", "other_gpu_busy", "limit_s"], rows)
    left = [r for r in rows if r["status"] not in ("ok", "failed")]
    write_csv("pending.csv", ["entry", "method", "status", "note"], [dict(entry=r["entry"], method=r["method"], status=r["status"], note=r["note"]) for r in left])
    summary_table(rows)
    log("\n%d tasks are finished (ok or failed), %d are left for the next session (pending.csv)" % (len(rows) - len(left), len(left)))
    STATE["complete"] = any(r["status"] in ("ok", "failed") for r in rows)
    if not STATE["complete"]:
        STATE["stopped_at"] = "no run finished (see the setup lines of each method)"
    STATE["pending"] = [[r["entry"], r["method"], r["status"]] for r in left]
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Last step. Pack the results

Run this cell last, also after a problem. It works after a restart of the kernel too: it then packs the files that are on disk.

In [ ]:
import os, sys
try:
    finish
except NameError:                                  # fresh kernel: nothing of the earlier cells is left, only the files on disk
    work = None
    for cand in (os.environ.get("MICA_WORK"), "/kaggle/working/pilot", "/content/pilot", os.path.abspath("pilot")):
        if cand and os.path.isfile(os.path.join(cand, "scripts", "nbtools.py")):
            work = cand
            break
    if work is None:
        print("RECOVERY: the work folder was not found, so there is nothing to pack. Run the notebook from the top.")
    else:
        sys.path.insert(0, os.path.join(work, "scripts"))
        import nbtools
        nbtools.recover("gpu_cryozeta_a")
else:
    zip_path = finish(bool(STATE["ok"] and STATE.get("complete")), {"methods": {k: {"label": m["label"], "commit": m["commit"], "status": m["status"], "setup_seconds": m["setup_seconds"], "version": m.get("version", "")} for k, m in STATE["methods"].items()}, "specs": globals().get("SPECS", {}), "usalign_commit": globals().get("USALIGN_COMMIT", ""), "usalign_version": STATE["usalign_version"], "input_sha256": globals().get("EXPECTED", {}), "tasks": globals().get("TASKS"), "hours_setting": globals().get("HOURS_SETTING"), "run_limit_s": globals().get("RUN_LIMIT"), "lanes": globals().get("LANES")})
    if STATE["ok"] and STATE.get("complete"):
        log("\nRESULT: OK. Download %s (Kaggle: the output files of this version) and send it to me." % zip_path)
    elif STATE["ok"]:
        log("\nRESULT: NOT COMPLETE (%s). Send me %s; it shows where it stopped." % (STATE.get("stopped_at") or "a step did not finish", zip_path))
    else:
        log("\nRESULT: STOPPED (%s). Send me %s." % (STATE["problem"], zip_path))
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        log("(Kaggle: open the version, then Output, then download the zip. Colab: the file list on the left.)")
